# Step 4.5 — Settle the Science (fair baselines + near-OOD verdict)

Self-contained: section **1b** writes all Step 4.5 code into the Drive repo, so nothing is missing. Run top-to-bottom.

## 1. Colab setup (mount Drive, install, set path)

In [1]:
# --- Colab setup ---------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

# Point this at your repo copy on Drive (same as the Step 4 notebook).
%cd /content/drive/MyDrive/bpeft_step4/thesis
!pip -q install -r requirements.txt
import sys; sys.path.insert(0, '.')

Mounted at /content/drive
/content/drive/MyDrive/bpeft_step4/thesis


## 1b. Materialize Step 4.5 code into the Drive repo

**Run once per session.** Each cell writes one file (`%%writefile`). No git, no manual uploads.

In [2]:
%%writefile configs/exp_phase2_evidential_retuned.yaml
# Step 4.5 / W2 — evidential retuned at the LOSS, not the affine.
#
# Story: Step 4's evidential head was worse-calibrated than softmax (ECE 0.344
# vs 0.119). The diagnosis (docs/superpowers/specs/2026-07-06-...): the
# aggressive evidence-affine (scale 4, bias -24) + KL-to-uniform (0.5) drove ID
# UNDER-confidence. R-EDL (Survey EDL summary) says the rigid "+1" prior and the
# variance-minimising regulariser can induce miscalibration and are droppable.
#
# These values are a STARTING POINT. The exact kl_weight_max / prior_per_class /
# use_variance are chosen by a VAL-only sweep (seeds 10000-10099) in
# notebooks/step4_5_settle.ipynb; they must NEVER be tuned on the 600 test seeds.
extends: exp_phase2_evidential.yaml

loss:
  kl_weight_max: 0.1       # lowered from 0.5 — less push toward the uniform prior
  kl_anneal_steps: 1000
  prior_per_class: 1.0     # R-EDL knob; sweep {1.0, 0.5} on val if ECE stays high
  use_variance: false      # R-EDL relaxation: drop the variance-minimising term

head:
  # Gentler affine than the (4, -24) sharpening; let the loss carry calibration.
  evidence_scale_init: 2.0
  evidence_bias_init: -6.0

wandb:
  tags: [phase2, step4_5, evidential, prototype, cifar_fs_bertinetto, redl]


Overwriting configs/exp_phase2_evidential_retuned.yaml


In [3]:
%%writefile src/losses/evidential.py
import torch


def kl_divergence_dirichlet(alpha: torch.Tensor, num_classes: int) -> torch.Tensor:
    """KL[Dir(alpha) || Dir(1,...,1)]. Sensoy 2018, Eq. 13. Returns shape (B,)."""
    ones = torch.ones_like(alpha)
    sum_alpha = alpha.sum(dim=-1, keepdim=True)
    K = torch.tensor(float(num_classes), device=alpha.device)
    return (
        torch.lgamma(sum_alpha).squeeze(-1)
        - torch.lgamma(K)
        - torch.lgamma(alpha).sum(dim=-1)
        + ((alpha - ones) * (torch.digamma(alpha) - torch.digamma(sum_alpha))).sum(dim=-1)
    )


def evidential_mse_loss(evidence: torch.Tensor, target_onehot: torch.Tensor,
                        num_classes: int, kl_weight: float,
                        *, prior_per_class: float = 1.0,
                        use_variance: bool = True) -> torch.Tensor:
    """Sensoy 2018 Eq. 5 + Eq. 13 KL prior, with R-EDL knobs (Survey EDL).

    prior_per_class: added Dirichlet mass per class (alpha = evidence +
      prior_per_class). 1.0 recovers Sensoy's rigid "+1" prior; smaller
      values give less prior mass (sharper mean, higher vacuity K/S).
    use_variance: include the Bayes-risk variance term (True = Sensoy;
      False = the R-EDL relaxation that drops the variance-minimising
      regulariser, which the Survey-EDL summary flags as a driver of
      miscalibration).

    Defaults (prior_per_class=1.0, use_variance=True) reproduce the
    original loss bit-for-bit. KL still only penalises wrong-class evidence.
    """
    alpha = evidence + float(prior_per_class)
    S = alpha.sum(dim=-1, keepdim=True)
    p = alpha / S

    mse_term = ((target_onehot - p) ** 2).sum(dim=-1)
    if use_variance:
        var = p * (1.0 - p) / (S + 1.0)
        mse_term = mse_term + var.sum(dim=-1)

    alpha_tilde = target_onehot + (1.0 - target_onehot) * alpha
    kl = kl_divergence_dirichlet(alpha_tilde, num_classes)

    return (mse_term + kl_weight * kl).mean()


Overwriting src/losses/evidential.py


In [4]:
%%writefile src/datasets/cifar_fs.py
"""CIFAR-FS few-shot dataset (Bertinetto 2019 64/16/20 split).

Step 4 (Phase 2) replaces the Step 1/2 stand-in (CIFAR-100 test, classes
0..19) with the proper Bertinetto split. The split is loaded from
data/cifar_fs_split.json as class NAMES; the loader converts names to
CIFAR-100 integer IDs at runtime using torchvision's `.classes` attribute.

The split file is canonical (do not regenerate). The committed file ships
with a synthetic alphabetical fallback and a `_status: synthetic_fallback`
flag — notebooks/step4_episodic.ipynb fetches the canonical Bertinetto
split and overwrites the file before any training run. `load_cifar_fs_split`
WARNS if `_status == 'synthetic_fallback'` so accidental training on the
fallback is loud, not silent.

Three asserts at load time:
  1. train ∩ val ∩ test == ∅  (no class appears in two splits)
  2. train ∪ val ∪ test == set(CIFAR-100)   (every class is in one split)
  3. |train| == 64, |val| == 16, |test| == 20  (matches the spec)

The function returned by `get_cifar_fs(split=...)` is a torchvision
CIFAR-100 dataset filtered to the split's class IDs, with labels
RE-INDEXED into a contiguous range:
  train -> 0..63, val -> 0..15, test -> 0..19.

This re-indexing is what lets the episode sampler treat the in-split
class IDs as a simple 0..N range without surprises.
"""
from __future__ import annotations
import json
import warnings
from pathlib import Path
from typing import Dict, List

import torch
from torch.utils.data import Dataset
from torchvision import datasets, transforms


_IMAGENET_MEAN = [0.485, 0.456, 0.406]
_IMAGENET_STD = [0.229, 0.224, 0.225]

_REPO_ROOT = Path(__file__).resolve().parents[2]
_SPLIT_PATH = _REPO_ROOT / "data" / "cifar_fs_split.json"


def _build_transform(image_size: int):
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=_IMAGENET_MEAN, std=_IMAGENET_STD),
    ])


def load_cifar_fs_split(split_path: Path | str | None = None,
                        cifar100_classes: List[str] | None = None,
                        ) -> Dict[str, List[int]]:
    """Load and validate the CIFAR-FS class split.

    Returns a dict with keys "train", "val", "test", values are sorted
    lists of CIFAR-100 integer class IDs (0..99).

    Args:
        split_path: optional explicit path to the split JSON. Defaults to
                    data/cifar_fs_split.json.
        cifar100_classes: optional list of 100 CIFAR-100 class names in
                    integer-ID order (i.e. cifar100_classes[i] is the name
                    of class i). If None, the canonical CIFAR-100 list is
                    used (built into this module).

    Raises:
        ValueError: if the split file is missing, malformed, or fails any
                    of the three structural assertions.
    """
    if cifar100_classes is None:
        cifar100_classes = CIFAR100_CLASS_NAMES
    name_to_id = {name: i for i, name in enumerate(cifar100_classes)}

    path = Path(split_path) if split_path else _SPLIT_PATH
    if not path.exists():
        raise ValueError(
            f"CIFAR-FS split file not found: {path}. "
            f"Run notebooks/step4_episodic.ipynb to fetch the canonical "
            f"Bertinetto split, or restore the synthetic fallback."
        )
    with open(path) as f:
        raw = json.load(f)

    if raw.get("_status") == "synthetic_fallback":
        warnings.warn(
            "CIFAR-FS split is the SYNTHETIC FALLBACK (not Bertinetto 2019). "
            "Run notebooks/step4_episodic.ipynb's 'fetch canonical split' "
            "cell before reporting Phase 2 results.",
            UserWarning,
            stacklevel=2,
        )

    out: Dict[str, List[int]] = {}
    for split_name, expected_count in [("train", 64), ("val", 16), ("test", 20)]:
        if split_name not in raw:
            raise ValueError(f"split file is missing required key {split_name!r}")
        names = list(raw[split_name])
        if len(names) != expected_count:
            raise ValueError(
                f"split {split_name!r} has {len(names)} entries, "
                f"expected {expected_count}"
            )
        try:
            ids = sorted(name_to_id[n] for n in names)
        except KeyError as e:
            raise ValueError(
                f"split {split_name!r} contains unknown CIFAR-100 class "
                f"name {e}. Allowed names: {sorted(name_to_id)[:5]}..."
            ) from e
        out[split_name] = ids

    train_set, val_set, test_set = (set(out[k]) for k in ("train", "val", "test"))
    overlap = (train_set & val_set) | (val_set & test_set) | (train_set & test_set)
    if overlap:
        raise ValueError(
            f"CIFAR-FS splits overlap on class IDs: {sorted(overlap)}"
        )
    union = train_set | val_set | test_set
    if union != set(range(100)):
        missing = set(range(100)) - union
        raise ValueError(
            f"CIFAR-FS splits do not cover all 100 CIFAR-100 classes; "
            f"missing IDs: {sorted(missing)}"
        )
    return out


class _RelabelledCIFAR100(Dataset):
    """Wraps CIFAR-100 (full) to only yield items whose original label is in
    `keep_ids`, AND maps the kept original labels into a contiguous local
    range [0..len(keep_ids)).

    The episode sampler then sees a tidy 0..N range on the local labels.
    """

    def __init__(self, base: Dataset, keep_ids: List[int]):
        self.base = base
        # CIFAR-100 exposes either `targets` (CIFAR100) or `labels`.
        if hasattr(base, "targets"):
            self._orig_targets = list(base.targets)
        elif hasattr(base, "labels"):
            self._orig_targets = list(base.labels)
        else:
            self._orig_targets = [int(base[i][1]) for i in range(len(base))]
        keep = set(int(c) for c in keep_ids)
        self._indices = [i for i, t in enumerate(self._orig_targets) if int(t) in keep]
        self._global_to_local = {c: i for i, c in enumerate(sorted(keep))}
        # Expose `targets` so the episode sampler's _label_index path works.
        self.targets = [
            self._global_to_local[int(self._orig_targets[i])] for i in self._indices
        ]

    def __len__(self) -> int:
        return len(self._indices)

    def __getitem__(self, i: int):
        x, _ = self.base[self._indices[i]]
        return x, self.targets[i]


def get_cifar_fs(data_root: str = "data", image_size: int = 224,
                 split: str = "test",
                 split_path: Path | str | None = None,
                 cifar100_classes: List[str] | None = None,
                 class_ids: List[int] | None = None) -> Dataset:
    """Return a CIFAR-FS Dataset filtered to one of the Bertinetto splits.

    split: "train" | "val" | "test"
       - "train" -> 64-class split, drawn from CIFAR-100's TRAIN partition
                    (more images per class)
       - "val"   -> 16-class split, drawn from CIFAR-100's TRAIN partition
       - "test"  -> 20-class split, drawn from CIFAR-100's TEST partition
                    (matches the Step 1-3 in-distribution evaluation pool)

    class_ids: optional override. If provided, the Bertinetto split file
       is ignored and the dataset is filtered to these literal CIFAR-100
       IDs. This is the backward-compat path used by Step 1-3's exp_step1
       configs (class_ids = [0..19]).
    """
    if split not in ("train", "val", "test"):
        raise ValueError(f"split must be train|val|test, got {split!r}")

    transform = _build_transform(image_size)
    use_train_partition = split in ("train", "val")
    # Pre-fetch with a browser User-Agent: the Toronto host 403s torchvision's
    # default downloader, which breaks fresh (gitignored) clones on Colab.
    from ._robust_download import ensure_archive
    ensure_archive(
        data_root, "cifar-100-python.tar.gz",
        ["https://www.cs.toronto.edu/~kriz/cifar-100-python.tar.gz",
         "http://www.cs.toronto.edu/~kriz/cifar-100-python.tar.gz"],
        extracted_dirname="cifar-100-python",
    )
    base = datasets.CIFAR100(
        root=data_root, train=use_train_partition,
        download=True, transform=transform,
    )

    if class_ids is not None:
        # Backwards-compat: legacy configs pass class_ids directly.
        return _RelabelledCIFAR100(base, list(class_ids))

    split_map = load_cifar_fs_split(
        split_path=split_path, cifar100_classes=cifar100_classes,
    )
    return _RelabelledCIFAR100(base, split_map[split])


def get_cifar_fs_heldout_ood(data_root: str = "data", image_size: int = 224,
                             num_samples: int = 500, seed: int = 42,
                             heldout_split: str = "val") -> torch.Tensor:
    """Zero-download near-OOD pool for the episodic evaluator.

    Samples ``num_samples`` images from the CIFAR-FS ``heldout_split`` classes
    (default "val" — the 16 val classes, disjoint from the 20 test-episode
    classes). Same visual domain as the ID data but novel classes → a clean,
    free near-OOD (OpenOOD near-OOD is "semantic shift only"). Returns
    (N, 3, image_size, image_size) ImageNet-normalized tensors, matching
    ``get_svhn_ood`` so the evaluator treats every OOD pool identically.
    """
    import random
    ds = get_cifar_fs(data_root=data_root, image_size=image_size,
                      split=heldout_split)
    rng = random.Random(seed)
    idx = rng.sample(range(len(ds)), min(num_samples, len(ds)))
    return torch.stack([ds[i][0] for i in idx])


# Canonical CIFAR-100 class names in integer-ID order.
# Source: torchvision.datasets.CIFAR100.classes (alphabetical).
# Frozen here so the loader does not require an instantiated CIFAR100
# object to do name->ID conversion.
CIFAR100_CLASS_NAMES: List[str] = [
    "apple", "aquarium_fish", "baby", "bear", "beaver",
    "bed", "bee", "beetle", "bicycle", "bottle",
    "bowl", "boy", "bridge", "bus", "butterfly",
    "camel", "can", "castle", "caterpillar", "cattle",
    "chair", "chimpanzee", "clock", "cloud", "cockroach",
    "couch", "crab", "crocodile", "cup", "dinosaur",
    "dolphin", "elephant", "flatfish", "forest", "fox",
    "girl", "hamster", "house", "kangaroo", "keyboard",
    "lamp", "lawn_mower", "leopard", "lion", "lizard",
    "lobster", "man", "maple_tree", "motorcycle", "mountain",
    "mouse", "mushroom", "oak_tree", "orange", "orchid",
    "otter", "palm_tree", "pear", "pickup_truck", "pine_tree",
    "plain", "plate", "poppy", "porcupine", "possum",
    "rabbit", "raccoon", "ray", "road", "rocket",
    "rose", "sea", "seal", "shark", "shrew",
    "skunk", "skyscraper", "snail", "snake", "spider",
    "squirrel", "streetcar", "sunflower", "sweet_pepper", "table",
    "tank", "telephone", "television", "tiger", "tractor",
    "train", "trout", "tulip", "turtle", "wardrobe",
    "whale", "willow_tree", "wolf", "woman", "worm",
]
assert len(CIFAR100_CLASS_NAMES) == 100
assert len(set(CIFAR100_CLASS_NAMES)) == 100


Overwriting src/datasets/cifar_fs.py


In [5]:
%%writefile src/datasets/tinyimagenet_ood.py
"""Near-OOD loader: TinyImageNet (OpenOOD near-OOD protocol; Step 7 plan).

Mirrors ``get_svhn_ood`` (same signature + ImageNet normalization) so the
episodic evaluator can treat every OOD pool uniformly. TinyImageNet is the
DECISIVE RQ3 test: far-OOD SVHN is easy for softmax-MSP, whereas evidential /
Dirichlet uncertainty is expected to differentiate on near-OOD where MSP
degrades (Malinin & Gales; OpenOOD near/far split).

PERFORMANCE (important on Colab+Drive): the archive holds ~120,000 tiny files.
Extracting or ImageFolder-scanning them on a Google-Drive FUSE mount does one
network round-trip PER FILE and can take HOURS. We therefore NEVER extract:
we cache the single 240MB zip (Drive-friendly: one big sequential file), copy
it once to local disk, and read only the ``num_samples`` images we actually
need straight out of the zip. This turns a multi-hour step into ~1-2 min.
"""
import io
import os
import random
import shutil
import tempfile
import zipfile

import torch
from PIL import Image
from torchvision import transforms

_IMAGENET_MEAN = [0.485, 0.456, 0.406]
_IMAGENET_STD = [0.229, 0.224, 0.225]
_TIN_URL = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"
_ZIP_NAME = "tiny-imagenet-200.zip"


def get_tinyimagenet_ood(data_root: str = "data", image_size: int = 224,
                         num_samples: int = 500, seed: int = 42) -> torch.Tensor:
    """Return (N, 3, image_size, image_size) ImageNet-normalized TinyImageNet
    images, sampled deterministically from the train partition's JPEGs. Reads
    the images directly from the zip (no extraction)."""
    transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=_IMAGENET_MEAN, std=_IMAGENET_STD),
    ])

    # Cache the zip once (Drive-friendly: a single large file downloads fast).
    from ._robust_download import ensure_archive
    ensure_archive(data_root, _ZIP_NAME, [_TIN_URL])
    drive_zip = os.path.join(data_root, _ZIP_NAME)

    # Read entries from a LOCAL copy: per-entry reads on a Drive mount are slow,
    # a single sequential copy to local disk is not. /content on Colab, else tmp.
    local_base = "/content" if os.path.isdir("/content") else tempfile.gettempdir()
    local_zip = os.path.join(local_base, _ZIP_NAME)
    if (not os.path.exists(local_zip)
            or os.path.getsize(local_zip) != os.path.getsize(drive_zip)):
        shutil.copyfile(drive_zip, local_zip)

    with zipfile.ZipFile(local_zip) as z:
        # train images live at tiny-imagenet-200/train/<wnid>/images/<f>.JPEG
        names = [n for n in z.namelist()
                 if "/train/" in n and n.lower().endswith(".jpeg")]
        names.sort()  # deterministic order before sampling
        if not names:
            raise RuntimeError(
                "No train JPEGs found in the TinyImageNet zip; the download may "
                "be corrupt — delete "
                f"{drive_zip} and re-run.")
        rng = random.Random(seed)
        pick = rng.sample(names, min(num_samples, len(names)))
        imgs = []
        for n in pick:
            raw = z.read(n)
            img = Image.open(io.BytesIO(raw)).convert("RGB")
            imgs.append(transform(img))

    return torch.stack(imgs)


Overwriting src/datasets/tinyimagenet_ood.py


In [6]:
%%writefile src/datasets/__init__.py
from .cifar_fs import (
    get_cifar_fs,
    get_cifar_fs_heldout_ood,
    load_cifar_fs_split,
    CIFAR100_CLASS_NAMES,
)
from .svhn_ood import get_svhn_ood
from .tinyimagenet_ood import get_tinyimagenet_ood
from .episode_sampler import sample_episode, EpisodicIterableDataset


def build_dataset(spec: dict):
    """spec: {name: cifar_fs, split: train|val|test, ...} | {name: svhn_ood, ...}.

    Step 4 (Phase 2) extends `cifar_fs` to read the Bertinetto 64/16/20
    split from data/cifar_fs_split.json. If `class_ids` is in the spec
    (legacy Step 1-3 path), it overrides the split file and filters to
    those literal CIFAR-100 class IDs.
    """
    name = spec["name"]
    if name == "cifar_fs":
        return get_cifar_fs(
            data_root=spec.get("data_root", "data"),
            image_size=spec.get("image_size", 224),
            split=spec.get("split", "test"),
            class_ids=spec.get("class_ids"),
        )
    raise ValueError(f"Unknown dataset: {name}")


__all__ = [
    "build_dataset",
    "get_cifar_fs",
    "get_cifar_fs_heldout_ood",
    "load_cifar_fs_split",
    "CIFAR100_CLASS_NAMES",
    "get_svhn_ood",
    "get_tinyimagenet_ood",
    "sample_episode",
    "EpisodicIterableDataset",
]


Overwriting src/datasets/__init__.py


In [7]:
%%writefile src/evaluators/temperature.py
"""Post-hoc temperature scaling (Guo et al. 2017).

Fits a single scalar T>0 that minimizes NLL of softmax(logits/T). Used as
the fair calibration baseline the evidential head must beat: T is fit ONCE
on the frozen validation episodes (seeds 10000-10099), then frozen and
applied to every test episode. T>0 does not move the argmax, so accuracy
is unchanged.

Reasoning (thesis instructions): source = Calibration summary (Guo et al.).
Pros: trivial, strong, the mandatory calibration baseline. Cons: needs a
val logit dump pass. Fit: directly answers "did you beat temperature
scaling?" — the biggest threat to the evidential-calibration claim.
Deviation from Guo: the "held-out validation set" is the frozen VAL
EPISODES (pooled query logits), the episodic analog of a fixed val set.
"""
from __future__ import annotations

import torch
import torch.nn.functional as F


def fit_temperature(logits: torch.Tensor, targets: torch.Tensor,
                    *, max_iter: int = 200, lr: float = 0.01) -> float:
    """Return scalar T>0 minimizing cross-entropy(softmax(logits/T), targets).

    Optimizes log_T (so T = exp(log_T) stays strictly positive) with Adam.
    """
    logits = logits.detach().float()
    targets = targets.detach().long()
    log_T = torch.zeros(1, requires_grad=True)  # T = exp(log_T) > 0
    opt = torch.optim.Adam([log_T], lr=lr)
    for _ in range(max_iter):
        opt.zero_grad()
        loss = F.cross_entropy(logits / log_T.exp(), targets)
        loss.backward()
        opt.step()
    return float(log_T.exp().item())


def apply_temperature(logits: torch.Tensor, T: float) -> torch.Tensor:
    """softmax(logits / T). T>0 preserves argmax (accuracy unchanged)."""
    return torch.softmax(logits / float(T), dim=-1)


Overwriting src/evaluators/temperature.py


In [8]:
%%writefile src/evaluators/ood.py
"""Out-of-distribution detection scoring.

Step 1-3 used ood_auroc only. Step 4 (Phase 2) adds FPR@95 — the false-
positive rate at the 95% true-positive-rate operating point — because
proposal §7 lists it under Reliability metrics.
"""
import numpy as np
import torch
from sklearn.metrics import roc_auc_score, roc_curve


def ood_auroc(id_scores: np.ndarray, ood_scores: np.ndarray) -> float:
    """In-distribution=1, OOD=0. Higher score should mean more in-distribution."""
    labels = np.concatenate([np.ones(len(id_scores)), np.zeros(len(ood_scores))])
    scores = np.concatenate([id_scores, ood_scores])
    return float(roc_auc_score(labels, scores))


def fpr_at_95_tpr(id_scores: np.ndarray, ood_scores: np.ndarray) -> float:
    """False positive rate (on OOD samples) at the threshold that gives
    95% true positive rate on in-distribution samples.

    In-distribution is the POSITIVE class (label = 1). A high "score"
    means the sample looks in-distribution. The threshold is set so
    that 95% of ID samples have score >= threshold; the FPR is the
    fraction of OOD samples that also pass that threshold.

    Lower is better. Reported in proposal §7 alongside AUROC.

    Edge cases:
      - if either input is empty, returns 1.0 (worst possible).
      - if no threshold reaches 95% TPR exactly, returns the FPR at the
        nearest threshold that achieves >= 0.95 TPR.
    """
    if len(id_scores) == 0 or len(ood_scores) == 0:
        return 1.0
    labels = np.concatenate([
        np.ones(len(id_scores)),
        np.zeros(len(ood_scores)),
    ])
    scores = np.concatenate([id_scores, ood_scores])
    fpr, tpr, _ = roc_curve(labels, scores)
    # roc_curve returns FPR/TPR ordered by decreasing threshold.
    # Find the smallest TPR >= 0.95; report its FPR.
    mask = tpr >= 0.95
    if not mask.any():
        return 1.0
    return float(fpr[mask][0])


def evidence_to_probs_and_vacuity(evidence: torch.Tensor, num_classes: int,
                                  prior_per_class: float = 1.0):
    """Convert non-negative evidence (B, K) to Dirichlet mean probs and
    vacuity = K/S. `evidence` is the head's OWN output (Linear+softplus
    EvidentialHead) OR softplus(prototype_logits) — the math is the same.

    `prior_per_class` is the R-EDL prior mass per class (alpha = evidence +
    prior_per_class); it MUST match the value used in the training loss so
    train-time and test-time Dirichlets agree. Default 1.0 = Sensoy "+1".
    """
    alpha = evidence + float(prior_per_class)
    S = alpha.sum(dim=-1, keepdim=True)
    probs = alpha / S
    vacuity = (num_classes / S).squeeze(-1)
    return probs, vacuity


def logits_to_probs_and_uncertainty(logits: torch.Tensor):
    """Softmax baseline: uncertainty = 1 - max_p."""
    probs = torch.softmax(logits, dim=-1)
    return probs, 1.0 - probs.max(dim=-1).values


def energy_score(logits: torch.Tensor, T: float = 1.0) -> torch.Tensor:
    """Energy-based ID-ness score (Liu et al. 2020, EBO). Returns
    ``T * logsumexp(logits / T)`` per sample — the NEGATIVE of the paper's
    energy E, so higher => more in-distribution (matches this module's
    "higher score = more ID" convention). Parameter-free at T=1.

    Reasoning (thesis instructions): strongest cheap logit-based OOD
    baseline; conceptual parallel to Dirichlet strength S (both measure
    "amount of support"). Added so the evidential vacuity signal is
    compared against a strong softmax-side OOD score, not just MSP.
    """
    return float(T) * torch.logsumexp(logits / float(T), dim=-1)


Overwriting src/evaluators/ood.py


In [9]:
%%writefile src/evaluators/episodic.py
"""Episodic evaluator for the Step 4 prototype-head meta-trained model.

Step 4.5 (W3) generalises this from a single OOD pool / single score to a
SCORE x OOD-POOL MATRIX so the decisive near-OOD comparison is fair:

  - multiple OOD pools (e.g. "svhn_far", "cifar100_near", "tin_near") passed
    as pre-computed backbone features;
  - multiple ID-ness scores per head:
      evidential -> {"vacuity"}                 (1 - u = 1 - K/S)
      softmax    -> {"msp", "energy"[, "ts_msp"]}
    where "ts_msp" is added when a temperature T (fit on the frozen VAL
    episodes, Guo et al.) is provided, and "energy" is Liu et al.'s logsumexp.

For each test episode the query set is prototype-classified against THIS
episode's support prototypes, and every OOD pool is scored against the SAME
prototypes (standard open-set few-shot protocol). AUROC + FPR@95 are
accumulated per (pool, score).

The returned `summary` keeps ALL Step-3/Step-4 keys (populated from a PRIMARY
pool + the head's native score, for back-compat) and ADDS:
  ood_auroc__{pool}__{score}, fpr_at_95_tpr__{pool}__{score}   (per cell)
  ece_ts, brier_ts                                             (softmax + T only)
"""
from __future__ import annotations
from collections import defaultdict
from typing import Dict, Iterable, Optional, Tuple

import numpy as np
import torch

from .accuracy import accuracy, f1_macro
from .calibration import expected_calibration_error, brier_score
from .ood import (
    ood_auroc,
    fpr_at_95_tpr,
    evidence_to_probs_and_vacuity,
    energy_score,
)
from .temperature import apply_temperature


def _proto_logits_for_query(model, support_feats: torch.Tensor,
                            support_y: torch.Tensor,
                            query_feats: torch.Tensor) -> torch.Tensor:
    """Prototype-similarity logits for `query_feats` using THIS episode's
    support set. Both feature tensors are backbone outputs (B, D); the model
    applies the adapter internally. Returns (Q, n_way) logits."""
    return model.forward_proto_from_features(support_feats, support_y, query_feats)


def _id_score_set(logits: torch.Tensor, interpretation: str, head,
                  num_classes: int, temperature: Optional[float],
                  prior_per_class: float) -> Dict[str, torch.Tensor]:
    """Map prototype logits to a dict {score_name: (B,) id-ness score}.
    Higher score => looks more in-distribution. `head.to_evidence` supplies
    the SAME logits->evidence map used at train time (no train/test skew)."""
    if interpretation == "evidential":
        evidence = head.to_evidence(logits)
        _, vacuity = evidence_to_probs_and_vacuity(
            evidence, num_classes, prior_per_class,
        )
        return {"vacuity": 1.0 - vacuity}
    if interpretation == "softmax":
        probs = torch.softmax(logits, dim=-1)
        scores = {
            "msp": probs.max(dim=-1).values,
            "energy": energy_score(logits),
        }
        if temperature is not None:
            scores["ts_msp"] = apply_temperature(logits, temperature).max(dim=-1).values
        return scores
    raise ValueError(f"Unknown interpretation: {interpretation!r}")


def _logits_to_probs(logits: torch.Tensor, num_classes: int,
                     interpretation: str, head,
                     prior_per_class: float) -> torch.Tensor:
    """Probability vector per sample (for ECE / Brier / Macro-F1)."""
    if interpretation == "evidential":
        evidence = head.to_evidence(logits)
        probs, _vac = evidence_to_probs_and_vacuity(
            evidence, num_classes, prior_per_class,
        )
        return probs
    if interpretation == "softmax":
        return torch.softmax(logits, dim=-1)
    raise ValueError(f"Unknown interpretation: {interpretation!r}")


def _native_score(interpretation: str) -> str:
    """The head's own OOD score, used to populate the legacy single-score keys."""
    return "vacuity" if interpretation == "evidential" else "msp"


def evaluate_episodic(
    model,
    test_iterable: Iterable[Tuple[torch.Tensor, torch.Tensor,
                                  torch.Tensor, torch.Tensor]],
    ood_pools: Optional[Dict[str, torch.Tensor]],
    *,
    num_classes: int,
    interpretation: str,            # "softmax" | "evidential"
    ece_bins: int = 15,
    temperature: Optional[float] = None,
    prior_per_class: float = 1.0,
    device: torch.device | str = "cpu",
    logger=None,
    wandb_run=None,
) -> dict:
    """Run prototype-head evaluation over a stream of test episodes.

    Args:
        ood_pools: {name: (N, D) backbone features}. Pass None/empty to skip OOD.
        interpretation: "softmax" or "evidential".
        temperature: optional T (softmax only) enabling the "ts_msp" score and
            the ece_ts / brier_ts calibration keys.
        prior_per_class: R-EDL Dirichlet prior mass per class; MUST match the
            training loss so train/test Dirichlets agree (default 1.0 = "+1").

    Returns dict {summary, pooled_probs, pooled_targets, last_id_scores,
    last_ood_scores}.
    """
    model = model.to(device)
    model.eval()
    backbone = model.backbone
    head = model.head

    ood_pools = ood_pools or {}
    ood_pools = {name: feats.to(device) for name, feats in ood_pools.items()}
    pool_names = list(ood_pools.keys())
    primary_pool = pool_names[0] if pool_names else None
    native = _native_score(interpretation)

    per_ep_acc, per_ep_f1, per_ep_ece, per_ep_brier = [], [], [], []
    pooled_probs, pooled_targets, pooled_logits = [], [], []
    # auroc[pool][score] -> list over episodes
    auroc_acc: Dict[str, Dict[str, list]] = defaultdict(lambda: defaultdict(list))
    fpr_acc: Dict[str, Dict[str, list]] = defaultdict(lambda: defaultdict(list))
    last_id_scores: np.ndarray | None = None
    last_ood_scores: np.ndarray | None = None

    with torch.no_grad():
        for i, (sx, sy, qx, qy) in enumerate(test_iterable):
            sx = sx.to(device); sy = sy.to(device)
            qx = qx.to(device); qy = qy.to(device)

            sx_feats = backbone(sx)
            qx_feats = backbone(qx)
            q_logits = _proto_logits_for_query(model, sx_feats, sy, qx_feats)

            probs = _logits_to_probs(q_logits, num_classes, interpretation,
                                     head, prior_per_class)
            per_ep_acc.append(accuracy(probs, qy))
            per_ep_f1.append(f1_macro(probs, qy, num_classes=num_classes))
            per_ep_ece.append(expected_calibration_error(probs, qy, num_bins=ece_bins))
            per_ep_brier.append(brier_score(probs, qy, num_classes))
            pooled_probs.append(probs.cpu())
            pooled_targets.append(qy.cpu())
            pooled_logits.append(q_logits.cpu())

            if pool_names:
                id_scores = _id_score_set(q_logits, interpretation, head,
                                          num_classes, temperature, prior_per_class)
                for pname in pool_names:
                    ood_logits = _proto_logits_for_query(
                        model, sx_feats, sy, ood_pools[pname])
                    ood_scores = _id_score_set(ood_logits, interpretation, head,
                                               num_classes, temperature, prior_per_class)
                    for sname, id_s in id_scores.items():
                        id_np = id_s.cpu().numpy()
                        ood_np = ood_scores[sname].cpu().numpy()
                        auroc_acc[pname][sname].append(ood_auroc(id_np, ood_np))
                        fpr_acc[pname][sname].append(fpr_at_95_tpr(id_np, ood_np))
                        if pname == primary_pool and sname == native:
                            last_id_scores, last_ood_scores = id_np, ood_np

            if logger is not None:
                prim = (auroc_acc[primary_pool][native][-1]
                        if primary_pool else float("nan"))
                logger.info(
                    f"ep {i:3d}  acc={per_ep_acc[-1]:.3f}  F1={per_ep_f1[-1]:.3f}  "
                    f"ECE={per_ep_ece[-1]:.3f}  Brier={per_ep_brier[-1]:.3f}  "
                    f"AUROC[{primary_pool}/{native}]={prim:.3f}"
                )
            if wandb_run is not None:
                wandb_run.log({
                    "eval/episode": i,
                    "eval/accuracy": float(per_ep_acc[-1]),
                    "eval/ECE": float(per_ep_ece[-1]),
                    "eval/accuracy_running_mean": float(np.mean(per_ep_acc)),
                })

    n = len(per_ep_acc)
    if n == 0:
        raise RuntimeError("evaluate_episodic: test_iterable was empty")

    pooled_probs_t = torch.cat(pooled_probs, dim=0)
    pooled_targets_t = torch.cat(pooled_targets, dim=0)
    pooled_logits_t = torch.cat(pooled_logits, dim=0)
    pooled_ece = expected_calibration_error(
        pooled_probs_t, pooled_targets_t, num_bins=ece_bins)

    def _ci95(arr):
        return float(1.96 * np.std(arr) / np.sqrt(len(arr))) if len(arr) else 0.0

    summary = {
        "accuracy_mean": float(np.mean(per_ep_acc)),
        "accuracy_std":  float(np.std(per_ep_acc)),
        "accuracy_ci95": _ci95(per_ep_acc),
        "f1_macro_mean": float(np.mean(per_ep_f1)),
        "f1_macro_std":  float(np.std(per_ep_f1)),
        "f1_macro_ci95": _ci95(per_ep_f1),
        "ece_per_episode_mean": float(np.mean(per_ep_ece)),
        "ece_per_episode_std":  float(np.std(per_ep_ece)),
        "ece_pooled":           float(pooled_ece),
        "brier_mean":           float(np.mean(per_ep_brier)),
        "brier_std":            float(np.std(per_ep_brier)),
        "num_episodes":         int(n),
        "interpretation":       str(interpretation),
    }

    # Per-cell matrix keys.
    for pname in pool_names:
        for sname, vals in auroc_acc[pname].items():
            summary[f"ood_auroc__{pname}__{sname}"] = float(np.mean(vals))
            summary[f"ood_auroc_std__{pname}__{sname}"] = float(np.std(vals))
            summary[f"fpr_at_95_tpr__{pname}__{sname}"] = float(np.mean(fpr_acc[pname][sname]))

    # Legacy single-score keys (primary pool + native score).
    if primary_pool is not None:
        summary["ood_auroc_mean"] = float(np.mean(auroc_acc[primary_pool][native]))
        summary["ood_auroc_std"]  = float(np.std(auroc_acc[primary_pool][native]))
        summary["fpr_at_95_tpr_mean"] = float(np.mean(fpr_acc[primary_pool][native]))
        summary["fpr_at_95_tpr_std"]  = float(np.std(fpr_acc[primary_pool][native]))
        summary["primary_ood_pool"] = str(primary_pool)
    else:
        summary["ood_auroc_mean"] = 0.0
        summary["ood_auroc_std"]  = 0.0
        summary["fpr_at_95_tpr_mean"] = 1.0
        summary["fpr_at_95_tpr_std"]  = 0.0

    # Temperature-scaled calibration (softmax only).
    if temperature is not None and interpretation == "softmax":
        ts_probs = apply_temperature(pooled_logits_t, temperature)
        summary["ece_ts"] = float(expected_calibration_error(
            ts_probs, pooled_targets_t, num_bins=ece_bins))
        summary["brier_ts"] = float(brier_score(ts_probs, pooled_targets_t, num_classes))

    return {
        "summary": summary,
        "pooled_probs": pooled_probs_t,
        "pooled_targets": pooled_targets_t,
        "last_id_scores": last_id_scores,
        "last_ood_scores": last_ood_scores,
    }


Overwriting src/evaluators/episodic.py


In [10]:
%%writefile src/evaluators/__init__.py
from .accuracy import accuracy, f1_macro
from .calibration import expected_calibration_error, brier_score
from .ood import (
    ood_auroc,
    fpr_at_95_tpr,
    evidence_to_probs_and_vacuity,
    logits_to_probs_and_uncertainty,
    energy_score,
)
from .temperature import fit_temperature, apply_temperature
from .episodic import evaluate_episodic

__all__ = [
    "accuracy",
    "f1_macro",
    "expected_calibration_error",
    "brier_score",
    "ood_auroc",
    "fpr_at_95_tpr",
    "evidence_to_probs_and_vacuity",
    "logits_to_probs_and_uncertainty",
    "energy_score",
    "fit_temperature",
    "apply_temperature",
    "evaluate_episodic",
]


Overwriting src/evaluators/__init__.py


In [11]:
%%writefile src/trainers/episodic_trainer.py
"""Episodic meta-training for B-PEFT (Step 4 / Phase 2).

Outer loop = epochs. Inner step = ONE episode.

Per episode:
  1. Sample (support, query) of (n_way × k_shot) + (n_way × q_query)
     images from the TRAIN split (Bertinetto 64 train classes).
  2. Forward both through frozen backbone -> trainable adapter ->
     parameter-free prototype head.
  3. Compute loss on query (cross-entropy for softmax interpretation,
     softplus -> evidential MSE+KL for evidential interpretation).
  4. Backprop. The adapter is the only thing that updates; the prototype
     head is parameter-free.

Per epoch:
  - episodes_per_epoch outer updates,
  - validation on val_episodes_per_epoch episodes from the VAL split
    (Bertinetto 16 val classes) using val_episodes.yaml's frozen seeds,
  - early stop on val accuracy plateau (no improvement for
    early_stop_patience epochs).

KL annealing for evidential mode: linear ramp 0 -> kl_weight_max over
kl_anneal_steps OUTER steps (1 outer step = 1 episode).

Smoke-collapse guard:
  After epoch 1 the validation accuracy MUST exceed 1/n_way + 0.05
  (the spec's R-EPISODIC-COLLAPSE guard, implementation.txt Step 4).
  Below that threshold the trainer raises EpisodicCollapse so Colab
  doesn't burn GPU on a doomed run.
"""
from __future__ import annotations
import copy
from dataclasses import dataclass, field
from typing import Iterable, List, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F


class EpisodicCollapse(RuntimeError):
    """Raised when validation accuracy after epoch 1 is at or below
    random-chance + 0.05. The spec's R-EPISODIC-COLLAPSE guard."""


def _one_hot(target: torch.Tensor, num_classes: int,
             dtype, device) -> torch.Tensor:
    return torch.eye(num_classes, dtype=dtype, device=device)[target]


@dataclass
class EpisodicHistory:
    """Per-epoch stats. The trainer dumps this into the checkpoint so
    the writeup / wandb summary can read it back without recomputing.
    """
    epoch:            List[int]            = field(default_factory=list)
    train_loss:       List[float]          = field(default_factory=list)
    train_acc:        List[float]          = field(default_factory=list)
    val_acc:          List[float]          = field(default_factory=list)
    val_loss:         List[float]          = field(default_factory=list)
    kl_weight_at_end: List[float]          = field(default_factory=list)
    # Instrumentation (adopted from the step4_reform diagnostics): per-epoch
    # mean Dirichlet evidence and adapter gradient norm. mean_evidence ~ 0 is
    # the fingerprint of the evidential collapse (softplus starved to zero);
    # adapter_grad_norm ~ 0 confirms no learning signal is reaching the PEFT
    # module. For softmax runs mean_evidence is logged as 0.0 (not applicable).
    mean_evidence:    List[float]          = field(default_factory=list)
    adapter_grad_norm: List[float]         = field(default_factory=list)


class EpisodicTrainer:
    """Outer epoch loop + per-episode forward + outer step + val + early
    stop. Intentionally KEEPS the same shape as the Step 1-3
    FewShotTrainer so scripts/train.py can branch on cfg.trainer.type
    without much extra wiring.
    """

    def __init__(
        self,
        model: nn.Module,
        optimizer: torch.optim.Optimizer,
        *,
        num_classes: int,
        num_epochs: int,
        episodes_per_epoch: int,
        val_episodes_per_epoch: int,
        early_stop_patience: int,
        interpretation: str,           # "softmax" | "evidential"
        kl_weight_max: Optional[float] = None,
        kl_anneal_steps: Optional[int] = None,
        ece_bins: int = 15,
        logger=None,
        wandb_run=None,
        device: torch.device | str = "cpu",
        collapse_threshold: float = 0.25,
        evid_prior_per_class: float = 1.0,
        evid_use_variance: bool = True,
    ):
        if interpretation not in ("softmax", "evidential"):
            raise ValueError(
                f"interpretation must be 'softmax' or 'evidential', got "
                f"{interpretation!r}"
            )
        if interpretation == "evidential" and (
            kl_weight_max is None or kl_anneal_steps is None
        ):
            raise ValueError(
                "evidential interpretation requires kl_weight_max + "
                "kl_anneal_steps"
            )

        self.model = model
        self.optimizer = optimizer
        self.num_classes = int(num_classes)
        self.num_epochs = int(num_epochs)
        self.episodes_per_epoch = int(episodes_per_epoch)
        self.val_episodes_per_epoch = int(val_episodes_per_epoch)
        self.early_stop_patience = int(early_stop_patience)
        self.interpretation = interpretation
        self.kl_weight_max = (None if kl_weight_max is None
                              else float(kl_weight_max))
        self.kl_anneal_steps = (None if kl_anneal_steps is None
                                else int(kl_anneal_steps))
        self.ece_bins = int(ece_bins)
        self.logger = logger
        self.wandb_run = wandb_run
        self.device = device
        self.collapse_threshold = float(collapse_threshold)
        # R-EDL knobs (Survey EDL): tunable prior mass + optional variance drop.
        # Defaults reproduce the Sensoy loss exactly; the retuned Phase-2 config
        # sweeps these on VAL to lower ID under-confidence / ECE.
        self.evid_prior_per_class = float(evid_prior_per_class)
        self.evid_use_variance = bool(evid_use_variance)

        self.history = EpisodicHistory()
        self.best_val_acc: float = -1.0
        self.best_val_epoch: int = -1
        self.best_state_dict: Optional[dict] = None

    # ------------------------------------------------------------------
    # Loss
    # ------------------------------------------------------------------
    def _episode_loss(self, q_logits: torch.Tensor, query_y: torch.Tensor,
                      global_step: int) -> torch.Tensor:
        if self.interpretation == "softmax":
            return F.cross_entropy(q_logits, query_y)

        # Evidential: logits -> evidence via the head's to_evidence (the
        # single source of truth shared with the evaluator, so the
        # softplus/recentre can never drift between train and test).
        evidence = self.model.head.to_evidence(q_logits)
        kl_w = (min(1.0, global_step / max(1, int(self.kl_anneal_steps)))
                * float(self.kl_weight_max))
        target_oh = _one_hot(
            query_y, self.num_classes, evidence.dtype, evidence.device,
        )
        # Reuse the existing Sensoy 2018 implementation.
        from ..losses.evidential import evidential_mse_loss
        return evidential_mse_loss(
            evidence, target_oh, num_classes=self.num_classes, kl_weight=kl_w,
            prior_per_class=self.evid_prior_per_class,
            use_variance=self.evid_use_variance,
        )

    def _kl_weight_at_step(self, step: int) -> float:
        if self.interpretation != "evidential":
            return 0.0
        return (min(1.0, step / max(1, int(self.kl_anneal_steps)))
                * float(self.kl_weight_max))

    # ------------------------------------------------------------------
    # Episode forward
    # ------------------------------------------------------------------
    def _forward_episode(self, sx, sy, qx):
        sx = sx.to(self.device); sy = sy.to(self.device)
        qx = qx.to(self.device)
        return self.model.forward_proto(sx, sy, qx)

    def _query_acc_from_logits(self, q_logits: torch.Tensor,
                               qy: torch.Tensor) -> float:
        preds = q_logits.argmax(dim=-1)
        return float((preds == qy.to(q_logits.device)).float().mean().item())

    # ------------------------------------------------------------------
    # Train / validate loops
    # ------------------------------------------------------------------
    def _adapter_grad_norm(self) -> float:
        """L2 norm over the adapter's gradients after backward(). A value
        of ~0 means no learning signal is reaching the PEFT module (the
        evidential-collapse fingerprint on the gradient side)."""
        sq = 0.0
        for p in self.model.adapter.parameters():
            if p.grad is not None:
                sq += float(p.grad.detach().pow(2).sum().item())
        return sq ** 0.5

    def _run_train_epoch(self, train_iter: Iterable,
                         global_step_start: int
                         ) -> tuple[float, float, float, float, int]:
        self.model.train()
        # Make sure frozen-BatchNorm in the backbone stays in eval mode
        # (same invariant as Step 1's FewShotTrainer.fit_episode).
        self.model.backbone.eval()
        for p in self.model.backbone.parameters():
            p.requires_grad = False

        total_loss = 0.0
        total_acc = 0.0
        total_evidence = 0.0
        total_grad_norm = 0.0
        n_eps = 0
        gs = global_step_start

        for sx, sy, qx, qy in train_iter:
            self.optimizer.zero_grad()
            q_logits = self._forward_episode(sx, sy, qx)
            loss = self._episode_loss(q_logits, qy.to(q_logits.device), gs)
            loss.backward()
            total_grad_norm += self._adapter_grad_norm()  # after backward, pre-step
            self.optimizer.step()

            # Mean Dirichlet evidence this episode (evidential only). Uses the
            # SAME head.to_evidence as the loss, so it tracks exactly what the
            # model trains on. softmax runs have no evidence -> log 0.0.
            if self.interpretation == "evidential":
                with torch.no_grad():
                    total_evidence += float(
                        self.model.head.to_evidence(q_logits).mean().item()
                    )

            total_loss += float(loss.item())
            total_acc  += self._query_acc_from_logits(q_logits, qy)
            n_eps += 1
            gs += 1

        d = max(1, n_eps)
        return (total_loss / d, total_acc / d,
                total_evidence / d, total_grad_norm / d, gs)

    @torch.no_grad()
    def _run_val_epoch(self, val_iter: Iterable) -> tuple[float, float]:
        self.model.eval()
        total_loss = 0.0
        total_acc = 0.0
        n_eps = 0
        for sx, sy, qx, qy in val_iter:
            q_logits = self._forward_episode(sx, sy, qx)
            # Use kl_weight=kl_weight_max for the val loss (no anneal at
            # val time). It's only used for monitoring; the early-stop
            # signal is val ACCURACY.
            if self.interpretation == "softmax":
                loss = F.cross_entropy(q_logits, qy.to(q_logits.device))
            else:
                evidence = self.model.head.to_evidence(q_logits)
                target_oh = _one_hot(
                    qy.to(evidence.device), self.num_classes,
                    evidence.dtype, evidence.device,
                )
                from ..losses.evidential import evidential_mse_loss
                loss = evidential_mse_loss(
                    evidence, target_oh, num_classes=self.num_classes,
                    kl_weight=float(self.kl_weight_max or 0.0),
                )
            total_loss += float(loss.item())
            total_acc  += self._query_acc_from_logits(q_logits, qy)
            n_eps += 1
        return total_loss / max(1, n_eps), total_acc / max(1, n_eps)

    # ------------------------------------------------------------------
    # Public entry point
    # ------------------------------------------------------------------
    def fit(self, train_iterable_factory, val_iterable_factory) -> dict:
        """Run the full episodic training schedule.

        Args:
            train_iterable_factory: a *callable* that returns a fresh
                training-episode iterator for the next epoch. The
                trainer calls this once per epoch so each epoch can use
                a different seed_offset and the episodes are NOT the
                same across epochs.
            val_iterable_factory:   same, for the validation stream.
                Typically called with a FIXED seed_offset so val
                comparisons across epochs are paired episode-for-
                episode.

        Returns:
            dict with: history (EpisodicHistory), best_val_acc,
            best_val_epoch, best_state_dict (the meta-trained adapter
            state at the best-validation epoch).
        """
        global_step = 0
        no_improve = 0
        for epoch in range(1, self.num_epochs + 1):
            tr_loss, tr_acc, tr_evidence, tr_grad_norm, global_step = \
                self._run_train_epoch(
                    train_iterable_factory(epoch), global_step,
                )
            val_loss, val_acc = self._run_val_epoch(
                val_iterable_factory(epoch),
            )
            kl_end = self._kl_weight_at_step(global_step)

            self.history.epoch.append(epoch)
            self.history.train_loss.append(tr_loss)
            self.history.train_acc.append(tr_acc)
            self.history.val_loss.append(val_loss)
            self.history.val_acc.append(val_acc)
            self.history.kl_weight_at_end.append(kl_end)
            self.history.mean_evidence.append(tr_evidence)
            self.history.adapter_grad_norm.append(tr_grad_norm)

            if self.logger is not None:
                self.logger.info(
                    f"epoch {epoch:3d}/{self.num_epochs}  "
                    f"train_loss={tr_loss:.4f}  train_acc={tr_acc:.3f}  "
                    f"val_loss={val_loss:.4f}  val_acc={val_acc:.3f}  "
                    f"kl_w={kl_end:.3f}  mean_ev={tr_evidence:.4f}  "
                    f"grad_norm={tr_grad_norm:.4f}  global_step={global_step}"
                )
            if self.wandb_run is not None:
                self.wandb_run.log({
                    "train/epoch": epoch,
                    "train/loss_epoch": tr_loss,
                    "train/acc_epoch":  tr_acc,
                    "val/loss":         val_loss,
                    "val/acc":          val_acc,
                    "train/kl_weight":  kl_end,
                    "train/mean_evidence":  tr_evidence,
                    "train/adapter_grad_norm": tr_grad_norm,
                    "train/global_step": global_step,
                }, step=epoch)

            # Collapse guard (R-EPISODIC-COLLAPSE), now two-sided:
            #   1. val accuracy at/below chance after epoch 1, OR
            #   2. evidential mean evidence ~ 0 after epoch 1 (the softplus-
            #      starvation fingerprint) -> the model cannot express
            #      confidence and no gradient flows. Abort before Colab burns
            #      a full session on a doomed run.
            if epoch == 1:
                if val_acc <= self.collapse_threshold:
                    raise EpisodicCollapse(
                        f"validation accuracy after epoch 1 was {val_acc:.3f}, "
                        f"<= collapse_threshold ({self.collapse_threshold}). "
                        f"Likely causes: KL anneal too aggressive, LR too "
                        f"high, or sampler / dataset mis-wired. Abort before "
                        f"burning more compute."
                    )
                if self.interpretation == "evidential" and tr_evidence <= 1e-3:
                    raise EpisodicCollapse(
                        f"mean Dirichlet evidence after epoch 1 was "
                        f"{tr_evidence:.2e} (~0): the evidence mapping is "
                        f"starved (softplus saturated to zero) so the model is "
                        f"a uniform-Dirichlet 'I don't know' for every input "
                        f"and no gradient reaches the adapter "
                        f"(grad_norm={tr_grad_norm:.2e}). Check head.metric / "
                        f"evidence_scale_init / evidence_bias_init."
                    )

            # Early stop on val acc plateau.
            if val_acc > self.best_val_acc + 1e-6:
                self.best_val_acc = val_acc
                self.best_val_epoch = epoch
                self.best_state_dict = copy.deepcopy(self.model.state_dict())
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= self.early_stop_patience:
                    if self.logger is not None:
                        self.logger.info(
                            f"early stop: no val improvement for "
                            f"{no_improve} epochs; "
                            f"best={self.best_val_acc:.3f} at "
                            f"epoch {self.best_val_epoch}"
                        )
                    break

        # Restore best weights so the caller's `model` is the one with
        # the best val accuracy (not the last-epoch one).
        if self.best_state_dict is not None:
            self.model.load_state_dict(self.best_state_dict)

        return {
            "history": self.history,
            "best_val_acc":   float(self.best_val_acc),
            "best_val_epoch": int(self.best_val_epoch),
            "best_state_dict": self.best_state_dict,
        }


Overwriting src/trainers/episodic_trainer.py


In [12]:
%%writefile scripts/build_cifar_fs_split.py
"""Materialize the canonical Bertinetto 2019 CIFAR-FS 64/16/20 split.

Source (authoritative, already used by notebooks/step4_episodic.ipynb): the
torchmeta mirror of the CIFAR-FS asset files. The original bertinetto/r2d2 repo
no longer hosts the lists. Each file is a JSON list of [superclass, class_name]
pairs; we keep only class_name and map it to a CIFAR-100 integer id via
torchvision's alphabetical class order (src/datasets/cifar_fs.CIFAR100_CLASS_NAMES).

Deterministic parse (NOT a summariser) so the 100 class names cannot be
silently dropped/altered. Validates: counts 64/16/20, pairwise-disjoint,
union == 0..99. Writes data/cifar_fs_split.json with
_status="canonical_bertinetto_via_torchmeta" — the SAME status label already
used by notebooks/step4_episodic.ipynb's fetch cell (same torchmeta source),
so a split fetched by either mechanism is recognized as canonical.

Run:  python scripts/build_cifar_fs_split.py
If network is unavailable, run notebooks/step4_episodic.ipynb's fetch cell
instead (it embeds the same canonical class names offline as a fallback).
"""
from __future__ import annotations

import json
import sys
import urllib.request
from pathlib import Path

_REPO_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(_REPO_ROOT))
from src.datasets.cifar_fs import CIFAR100_CLASS_NAMES  # noqa: E402

_BASE = ("https://raw.githubusercontent.com/tristandeleu/pytorch-meta/"
         "master/torchmeta/datasets/assets/cifar100/cifar-fs")
_SPLIT_FILES = {"train": f"{_BASE}/train.json",
                "val":   f"{_BASE}/val.json",
                "test":  f"{_BASE}/test.json"}
_OUT = _REPO_ROOT / "data" / "cifar_fs_split.json"


def _fetch_names(url: str) -> list[str]:
    with urllib.request.urlopen(url, timeout=30) as r:
        pairs = json.loads(r.read().decode("utf-8"))
    # Each entry is [superclass, class_name]; keep class_name only.
    return [p[1] for p in pairs]


def main() -> None:
    name_to_id = {n: i for i, n in enumerate(CIFAR100_CLASS_NAMES)}
    split_names = {k: _fetch_names(u) for k, u in _SPLIT_FILES.items()}

    counts = {"train": 64, "val": 16, "test": 20}
    for split, names in split_names.items():
        if len(names) != counts[split]:
            raise SystemExit(f"{split}: got {len(names)} names, expected {counts[split]}")
        unknown = [n for n in names if n not in name_to_id]
        if unknown:
            raise SystemExit(f"{split}: unknown CIFAR-100 class names {unknown}")

    ids = {k: sorted(name_to_id[n] for n in v) for k, v in split_names.items()}
    tr, va, te = (set(ids[k]) for k in ("train", "val", "test"))
    if (tr & va) or (va & te) or (tr & te):
        raise SystemExit("splits overlap on class ids")
    if (tr | va | te) != set(range(100)):
        raise SystemExit("splits do not cover all 100 CIFAR-100 ids")

    # Status string matches the convention already established by
    # notebooks/step4_episodic.ipynb's fetch cell (same torchmeta source) —
    # NOT a new invented label — so a split fetched by either mechanism is
    # recognized as canonical by anything checking `_status`.
    status = "canonical_bertinetto_via_torchmeta"
    out = {
        "_comment": "CIFAR-FS class split (Bertinetto 2019), class NAMES; "
                    "loader maps names -> ids via torchvision CIFAR-100 order.",
        "_canonical_source": _BASE,
        "_status": status,
        "_freeze_after_fetch": "FROZEN — do not regenerate.",
        "train": split_names["train"],
        "val":   split_names["val"],
        "test":  split_names["test"],
    }
    with open(_OUT, "w") as f:
        json.dump(out, f, indent=2)
    print(f"wrote {_OUT}  (64/16/20, disjoint, union=100, status={status})")


if __name__ == "__main__":
    main()


Overwriting scripts/build_cifar_fs_split.py


In [13]:
%%writefile scripts/train.py
"""Train a B-PEFT model from a YAML config.

Two modes, dispatched on cfg.trainer.type:

  trainer.type: single_episode    (Step 1-3 legacy)
      One episode of (n_way, k_shot, q_query), 200 inner training steps,
      LinearHead / EvidentialHead. Used by configs/exp_step1.yaml +
      configs/exp_step1_softmax.yaml — the Step 3 reproduction tests
      depend on this path being identical.

  trainer.type: episodic          (Step 4 / Phase 2)
      Episodic meta-training: per epoch we sample `episodes_per_epoch`
      independent episodes from the Bertinetto TRAIN split (64 classes),
      apply one outer gradient step per episode, then validate on the
      VAL split (16 classes) using configs/val_episodes.yaml. Early stop
      on val accuracy plateau. The adapter is the trainable PEFT
      module; the head is a parameter-free PrototypeHead.

Usage:
    python scripts/train.py --config configs/exp_step1.yaml          # legacy
    python scripts/train.py --config configs/exp_phase2_evidential.yaml
    python scripts/train.py --config configs/exp_phase2_softmax.yaml
"""
from __future__ import annotations
import argparse
import json
import sys
from pathlib import Path

# Make `src` importable when this script is run directly.
sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

import torch
import yaml

from src.utils import (
    set_seed, get_device, count_trainable_params, load_config, get_logger,
    WandbRun, make_run_name,
)
from src.datasets import (
    build_dataset, sample_episode, get_cifar_fs, EpisodicIterableDataset,
)
from src.models import build_model
from src.losses import build_loss
from src.trainers import train_one_episode, EpisodicTrainer


# =====================================================================
# Shared helpers
# =====================================================================
def _build_wandb_run(cfg, args, head_type, job_type: str = "train",
                    extra_tags: list[str] | None = None) -> WandbRun:
    """Open a W&B run for this training job. Honours --wandb-mode CLI flag."""
    wcfg = cfg.get("wandb", None) if isinstance(cfg, dict) else None
    project = (wcfg or {}).get("project", "bpeft-thesis") if wcfg else "bpeft-thesis"
    base_mode = (wcfg or {}).get("mode", "online") if wcfg else "online"
    disabled = bool((wcfg or {}).get("disabled", False)) if wcfg else False
    group = (wcfg or {}).get("group", None) if wcfg else None
    tags = list((wcfg or {}).get("tags", []) or []) if wcfg else []
    if extra_tags:
        tags = tags + list(extra_tags)

    mode = args.wandb_mode or base_mode
    if args.wandb_mode == "disabled":
        disabled = True
    return WandbRun(
        project=project,
        run_name=make_run_name(cfg),
        config={
            "config_path": str(Path(args.config).resolve()),
            "seed": int(cfg.seed),
            "head": dict(cfg.head),
            "adapter": dict(cfg.adapter),
            "loss": dict(cfg.loss),
            "trainer": dict(cfg.trainer),
            "dataset": {k: v for k, v in dict(cfg.dataset).items()
                        if k != "class_ids"},
            "train": dict(cfg.train),
        },
        mode=mode, disabled=disabled, group=group,
        tags=tags + [f"head:{head_type}", f"adapter:{cfg.adapter.type}"],
        job_type=job_type,
    )


def _head_descriptor(cfg) -> str:
    """Filesystem-safe head identifier.

    Step 1-3: head.type is sufficient (softmax / evidential).
    Step 4+:  prototype configs need head.interpretation appended,
              otherwise the two Phase 2 configs collide on disk.
    """
    head_type = cfg.head.type
    if head_type == "prototype":
        return f"prototype-{cfg.head.get('interpretation', 'evidential')}"
    return head_type


def _checkpoint_tag(cfg) -> str:
    """Filesystem-safe tag identifying this training run."""
    return f"{cfg.adapter.type}_{_head_descriptor(cfg)}_seed{cfg.seed}"


# =====================================================================
# Step 1-3 legacy path (trainer.type: single_episode)
# =====================================================================
def _train_single_episode(cfg, args, logger, device, wb) -> None:
    """Existing Step 1-3 single-episode training flow. Unchanged on
    purpose so the Step 3 reproduction tests stay byte-identical."""
    head_type = cfg.head.type

    dataset = build_dataset(dict(cfg.dataset))
    support_x, support_y, query_x, query_y = sample_episode(
        dataset=dataset,
        class_ids=list(cfg.dataset.class_ids) if cfg.dataset.get("class_ids") else None,
        n_way=int(cfg.dataset.n_way),
        k_shot=int(cfg.dataset.k_shot),
        q_query=int(cfg.dataset.q_query),
        seed=int(cfg.seed),
    )
    support_x = support_x.to(device)
    support_y = support_y.to(device)

    model = build_model(cfg).to(device)
    n_params = count_trainable_params(model)
    logger.info(f"trainable params: {n_params:,}")
    wb.update_summary({"n_params": int(n_params)})

    with torch.no_grad():
        support_feats = model.backbone(support_x)

    loss_spec = dict(cfg.loss)
    loss_spec["type"] = "evidential" if head_type == "evidential" else "cross_entropy"
    loss_fn = build_loss(loss_spec)

    history = train_one_episode(
        model=model,
        support_x=support_feats,
        support_y=support_y,
        loss_fn=loss_fn,
        num_classes=int(cfg.dataset.n_way),
        head_type=head_type,
        lr=float(cfg.train.lr),
        weight_decay=float(cfg.train.weight_decay),
        num_steps=int(cfg.train.num_steps),
        log_every=int(cfg.train.log_every),
        logger=logger,
        wandb_run=wb,
        kl_weight_max=float(cfg.loss.kl_weight_max) if head_type == "evidential" else None,
        kl_anneal_steps=int(cfg.loss.kl_anneal_steps) if head_type == "evidential" else None,
    )

    ckpt_dir = Path(cfg.output.checkpoint_dir)
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    tag = _checkpoint_tag(cfg)
    ckpt_path = ckpt_dir / f"model_{tag}.pt"
    torch.save({
        "state_dict": model.state_dict(),
        "config_path": str(Path(args.config).resolve()),
        "head_type": head_type,
        "adapter_type": cfg.adapter.type,
        "trainer_type": "single_episode",
        "train_history": history,
        "episode": {
            "support_x": support_x.cpu(),
            "support_y": support_y.cpu(),
            "query_x": query_x,
            "query_y": query_y,
        },
    }, ckpt_path)
    logger.info(f"saved checkpoint: {ckpt_path}")

    final_acc = history["support_acc"][-1]
    final_loss = history["loss"][-1]
    logger.info(f"final support accuracy: {final_acc:.3f}")
    wb.update_summary({
        "train/final_support_acc": float(final_acc),
        "train/final_loss": float(final_loss),
    })


# =====================================================================
# Step 4 / Phase 2 path (trainer.type: episodic)
# =====================================================================
def _train_episodic(cfg, args, logger, device, wb) -> None:
    """Episodic meta-training with prototype head."""
    interp = cfg.head.get("interpretation", "evidential")
    if interp not in ("softmax", "evidential"):
        raise ValueError(
            f"head.interpretation must be 'softmax' or 'evidential' for "
            f"trainer.type=episodic; got {interp!r}"
        )
    if cfg.head.type != "prototype":
        raise ValueError(
            f"trainer.type=episodic requires head.type=prototype; "
            f"got head.type={cfg.head.type!r}"
        )

    # --- Datasets: train + val splits ---------------------------------
    train_split = get_cifar_fs(
        data_root=cfg.dataset.data_root,
        image_size=int(cfg.dataset.image_size),
        split="train",
    )
    val_split = get_cifar_fs(
        data_root=cfg.dataset.data_root,
        image_size=int(cfg.dataset.image_size),
        split="val",
    )
    n_way   = int(cfg.dataset.n_way)
    k_shot  = int(cfg.dataset.k_shot)
    q_query = int(cfg.dataset.q_query)
    eps_per_epoch    = int(cfg.trainer.episodes_per_epoch)
    val_eps_per_epoch = int(cfg.trainer.val_episodes_per_epoch)
    train_seed_offset = int(cfg.trainer.train_seed_offset)

    # Frozen val seeds (configs/val_episodes.yaml).
    repo_root = Path(__file__).resolve().parents[1]
    with open(repo_root / cfg.eval.val_episodes_file) as f:
        val_eps_spec = yaml.safe_load(f)
    val_seeds = list(val_eps_spec["seeds"])
    if len(val_seeds) < val_eps_per_epoch:
        raise ValueError(
            f"val_episodes.yaml has {len(val_seeds)} seeds but "
            f"val_episodes_per_epoch={val_eps_per_epoch}"
        )
    val_seed_offset = int(val_seeds[0])

    def train_iter_factory(epoch: int):
        # Different seed_offset each epoch so train episodes don't repeat
        # across epochs; deterministic given (TRAIN_BASE, epoch, eps_per_epoch).
        offset = train_seed_offset + (epoch - 1) * eps_per_epoch
        return EpisodicIterableDataset(
            train_split, n_way=n_way, k_shot=k_shot, q_query=q_query,
            num_episodes=eps_per_epoch, seed_offset=offset,
        )

    def val_iter_factory(epoch: int):
        # Fixed seed offset so the same val episodes are used every epoch.
        # That makes the early-stop comparison paired.
        return EpisodicIterableDataset(
            val_split, n_way=n_way, k_shot=k_shot, q_query=q_query,
            num_episodes=val_eps_per_epoch, seed_offset=val_seed_offset,
        )

    # --- Model + optimiser --------------------------------------------
    model = build_model(cfg).to(device)
    n_params = count_trainable_params(model)
    logger.info(f"trainable params: {n_params:,}")
    wb.update_summary({"n_params": int(n_params)})

    optimizer = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad],
        lr=float(cfg.train.lr),
        weight_decay=float(cfg.train.weight_decay),
    )

    trainer = EpisodicTrainer(
        model=model,
        optimizer=optimizer,
        num_classes=n_way,
        num_epochs=int(cfg.trainer.num_epochs),
        episodes_per_epoch=eps_per_epoch,
        val_episodes_per_epoch=val_eps_per_epoch,
        early_stop_patience=int(cfg.trainer.early_stop_patience),
        interpretation=interp,
        kl_weight_max=(float(cfg.loss.kl_weight_max)
                        if interp == "evidential" else None),
        kl_anneal_steps=(int(cfg.loss.kl_anneal_steps)
                         if interp == "evidential" else None),
        ece_bins=int(cfg.eval.ece_bins),
        logger=logger,
        wandb_run=wb,
        device=device,
        collapse_threshold=float(cfg.trainer.collapse_threshold),
        # R-EDL knobs (Step 4.5 / W2); defaults reproduce the Sensoy loss.
        evid_prior_per_class=float(cfg.loss.get("prior_per_class", 1.0)),
        evid_use_variance=bool(cfg.loss.get("use_variance", True)),
    )
    result = trainer.fit(train_iter_factory, val_iter_factory)

    # --- Save ----------------------------------------------------------
    ckpt_dir = Path(cfg.output.checkpoint_dir)
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    tag = _checkpoint_tag(cfg)
    ckpt_path = ckpt_dir / f"model_phase2_{tag}.pt"
    history = result["history"]
    torch.save({
        "state_dict":   model.state_dict(),
        "config_path":  str(Path(args.config).resolve()),
        "head_type":    cfg.head.type,
        "adapter_type": cfg.adapter.type,
        "trainer_type": "episodic",
        "interpretation": interp,
        "best_val_acc":   result["best_val_acc"],
        "best_val_epoch": result["best_val_epoch"],
        "train_history": {
            "epoch":             list(history.epoch),
            "train_loss":        list(history.train_loss),
            "train_acc":         list(history.train_acc),
            "val_loss":          list(history.val_loss),
            "val_acc":           list(history.val_acc),
            "kl_weight_at_end":  list(history.kl_weight_at_end),
            "mean_evidence":     list(history.mean_evidence),
            "adapter_grad_norm": list(history.adapter_grad_norm),
        },
    }, ckpt_path)
    logger.info(
        f"saved checkpoint: {ckpt_path}  "
        f"best_val_acc={result['best_val_acc']:.3f} at "
        f"epoch {result['best_val_epoch']}"
    )
    wb.update_summary({
        "train/best_val_acc":   float(result["best_val_acc"]),
        "train/best_val_epoch": int(result["best_val_epoch"]),
        "train/total_epochs":   int(history.epoch[-1] if history.epoch else 0),
    })


# =====================================================================
# main
# =====================================================================
def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True, help="Path to experiment YAML")
    parser.add_argument("--seed-override", type=int, default=None,
                        help="Override cfg.seed (used by run_grid.sh)")
    parser.add_argument("--wandb-mode", choices=["online", "offline", "disabled"],
                        default=None, help="Override cfg.wandb.mode")
    args = parser.parse_args()

    cfg = load_config(args.config)
    if args.seed_override is not None:
        cfg.seed = args.seed_override
    logger = get_logger("bpeft.train")
    trainer_type = cfg.get("trainer", {}).get("type", "single_episode") if isinstance(cfg, dict) else "single_episode"
    logger.info(f"config: {args.config}  seed: {cfg.seed}  trainer.type: {trainer_type}")

    set_seed(int(cfg.seed))
    device = get_device()
    head_type = cfg.head.type

    extra_tags = [f"trainer:{trainer_type}",
                  f"phase:{'2' if trainer_type == 'episodic' else '1'}"]
    wb = _build_wandb_run(cfg, args, head_type, extra_tags=extra_tags)
    if not wb.disabled:
        logger.info(f"wandb run: {wb.run_name}  ({wb.mode})  url={wb.url}")
    else:
        logger.info("wandb: disabled (no-op logger)")

    if trainer_type == "single_episode":
        _train_single_episode(cfg, args, logger, device, wb)
    elif trainer_type == "episodic":
        _train_episodic(cfg, args, logger, device, wb)
    else:
        raise ValueError(
            f"Unknown trainer.type: {trainer_type!r}; expected "
            f"'single_episode' or 'episodic'."
        )

    wb.finish()


if __name__ == "__main__":
    main()


Overwriting scripts/train.py


In [14]:
%%writefile scripts/evaluate.py
"""Evaluate a trained B-PEFT checkpoint over a fixed list of test episodes.

Two modes, dispatched on cfg.trainer.type:

  trainer.type: single_episode    (Step 1-3 legacy)
      Per-episode fine-tune protocol: build a fresh model each episode,
      train on the support set for 200 inner steps, evaluate on the
      query set. The 600 test seeds come from configs/test_episodes.yaml.

  trainer.type: episodic          (Step 4 / Phase 2)
      Episodic test protocol: load the meta-trained adapter checkpoint
      (frozen), prototype-classify each episode's query set, score OOD
      samples against THIS episode's prototypes. Same 600 seeds, no
      inner fitting.

Both modes:
  - read the canonical seed list from configs/test_episodes.yaml,
  - support --num-episodes K to truncate to the first K seeds,
  - dump a metrics JSON with sort_keys=True for byte-identical reruns,
  - log to W&B (online | offline | disabled).

Usage:
    python scripts/evaluate.py --config configs/exp_step1.yaml          # legacy
    python scripts/evaluate.py --config configs/exp_phase2_evidential.yaml
"""
from __future__ import annotations
import argparse
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

import numpy as np
import torch
import yaml

from src.utils import (
    set_seed, get_device, count_trainable_params, load_config, get_logger,
    WandbRun, make_run_name, reliability_diagram, ood_histogram, confusion_matrix,
)
from src.datasets import (
    build_dataset, sample_episode, get_svhn_ood, get_cifar_fs,
    get_cifar_fs_heldout_ood, get_tinyimagenet_ood,
    EpisodicIterableDataset,
)
from src.models import build_model
from src.losses import build_loss
from src.trainers import train_one_episode
from src.evaluators import (
    accuracy, f1_macro, expected_calibration_error, brier_score,
    ood_auroc, fpr_at_95_tpr, energy_score, fit_temperature,
    evidence_to_probs_and_vacuity, logits_to_probs_and_uncertainty,
    evaluate_episodic,
)


def _head_descriptor(cfg) -> str:
    """Same as scripts/train.py:_head_descriptor — kept in sync.

    Step 1-3: head.type is sufficient (softmax / evidential).
    Step 4+:  prototype configs need head.interpretation appended,
              otherwise the two Phase 2 configs would collide on disk.
    """
    head_type = cfg.head.type
    if head_type == "prototype":
        return f"prototype-{cfg.head.get('interpretation', 'evidential')}"
    return head_type


# =====================================================================
# Shared helpers
# =====================================================================
def _extract_features(backbone, x, device, batch_size=64):
    """Run the frozen backbone once over `x` and return (N, D) features on CPU."""
    backbone.eval()
    chunks = []
    with torch.no_grad():
        for i in range(0, len(x), batch_size):
            batch = x[i:i + batch_size].to(device)
            chunks.append(backbone(batch).cpu())
    return torch.cat(chunks, dim=0)


def _predict_from_features(model, feats, head_type, device,
                           batch_size=256, num_classes=5):
    """Step 1-3 single-episode path: model with LinearHead / EvidentialHead."""
    model.eval()
    out_chunks = []
    with torch.no_grad():
        for i in range(0, len(feats), batch_size):
            batch = feats[i:i + batch_size].to(device)
            out_chunks.append(model.forward_from_features(batch).cpu())
    output = torch.cat(out_chunks, dim=0)
    if head_type == "evidential":
        return evidence_to_probs_and_vacuity(output, num_classes)
    return logits_to_probs_and_uncertainty(output)


def _load_test_seeds(repo_root: Path, cfg) -> list[int]:
    eps_path = repo_root / cfg.eval.episodes_file
    with open(eps_path) as f:
        spec = yaml.safe_load(f)
    seeds = list(spec["seeds"])
    assert len(seeds) == int(spec["num_episodes"]), \
        f"{eps_path}: num_episodes ({spec['num_episodes']}) != len(seeds) ({len(seeds)})"
    return seeds


def _build_wandb_run(cfg, args, head_type, extra_tags=None) -> WandbRun:
    wcfg = cfg.get("wandb", None) if isinstance(cfg, dict) else None
    project = (wcfg or {}).get("project", "bpeft-thesis") if wcfg else "bpeft-thesis"
    base_mode = (wcfg or {}).get("mode", "online") if wcfg else "online"
    disabled = bool((wcfg or {}).get("disabled", False)) if wcfg else False
    group = (wcfg or {}).get("group", None) if wcfg else None
    tags = list((wcfg or {}).get("tags", []) or []) if wcfg else []
    if extra_tags:
        tags = tags + list(extra_tags)

    mode = args.wandb_mode or base_mode
    if args.wandb_mode == "disabled":
        disabled = True
    return WandbRun(
        project=project,
        run_name=make_run_name(cfg) + "_eval",
        config={
            "config_path": str(Path(args.config).resolve()),
            "seed": int(cfg.seed),
            "head": dict(cfg.head),
            "adapter": dict(cfg.adapter),
            "loss": dict(cfg.loss),
            "trainer": dict(cfg.trainer),
            "eval": dict(cfg.eval),
            "ood": dict(cfg.ood),
        },
        mode=mode, disabled=disabled, group=group,
        tags=tags + [f"head:{head_type}", f"adapter:{cfg.adapter.type}"],
        job_type="evaluate",
    )


# =====================================================================
# Step 1-3 legacy path (per-episode fine-tune)
# =====================================================================
def _evaluate_finetune(cfg, args, logger, device, wb, seeds, repo_root) -> dict:
    """Existing Step 3 per-episode fine-tune evaluator. Verbatim from the
    pre-Step-4 version of this script."""
    n_eval = len(seeds)
    set_seed(int(cfg.seed))
    head_type = cfg.head.type
    K = int(cfg.dataset.n_way)

    dataset = build_dataset(dict(cfg.dataset))
    svhn_x = get_svhn_ood(
        data_root=cfg.ood.data_root,
        image_size=int(cfg.dataset.image_size),
        num_samples=int(cfg.ood.num_samples),
        seed=int(cfg.ood.seed),
    )
    shared_model = build_model(cfg).to(device)
    svhn_feats = _extract_features(shared_model.backbone, svhn_x, device)
    logger.info(f"cached SVHN features: {tuple(svhn_feats.shape)}")

    per_ep_acc, per_ep_ece, per_ep_brier, per_ep_auroc = [], [], [], []
    pooled_probs, pooled_targets = [], []
    last_id_scores = None
    last_ood_scores = None

    for i, ep_seed in enumerate(seeds):
        set_seed(int(cfg.seed) + int(ep_seed))
        model = build_model(cfg).to(device)
        loss_spec = dict(cfg.loss)
        loss_spec["type"] = "evidential" if head_type == "evidential" else "cross_entropy"
        loss_fn = build_loss(loss_spec)

        support_x, support_y, query_x, query_y = sample_episode(
            dataset=dataset,
            class_ids=list(cfg.dataset.class_ids) if cfg.dataset.get("class_ids") else None,
            n_way=K, k_shot=int(cfg.dataset.k_shot),
            q_query=int(cfg.dataset.q_query),
            seed=int(ep_seed),
        )
        support_feats = _extract_features(model.backbone, support_x, device).to(device)
        query_feats   = _extract_features(model.backbone, query_x, device)
        support_y_d = support_y.to(device)

        train_one_episode(
            model=model,
            support_x=support_feats, support_y=support_y_d,
            loss_fn=loss_fn, num_classes=K, head_type=head_type,
            lr=float(cfg.train.lr), weight_decay=float(cfg.train.weight_decay),
            num_steps=int(cfg.train.num_steps),
            log_every=10**9, logger=None, wandb_run=None,
        )

        probs, vac = _predict_from_features(model, query_feats, head_type, device, num_classes=K)
        acc = accuracy(probs, query_y)
        ece = expected_calibration_error(probs, query_y, num_bins=int(cfg.eval.ece_bins))
        bri = brier_score(probs, query_y, K)

        _, ood_vac = _predict_from_features(model, svhn_feats, head_type, device, num_classes=K)
        id_scores  = (1.0 - vac).numpy()
        ood_scores = (1.0 - ood_vac).numpy()
        auroc = ood_auroc(id_scores, ood_scores)
        last_id_scores, last_ood_scores = id_scores, ood_scores

        per_ep_acc.append(acc); per_ep_ece.append(ece)
        per_ep_brier.append(bri); per_ep_auroc.append(auroc)
        pooled_probs.append(probs); pooled_targets.append(query_y)
        logger.info(
            f"ep {i:3d} (seed={ep_seed:3d})  acc={acc:.3f}  ECE={ece:.3f}  "
            f"Brier={bri:.3f}  AUROC={auroc:.3f}"
        )
        wb.log({
            "eval/episode": i,
            "eval/episode_seed": int(ep_seed),
            "eval/accuracy": float(acc),
            "eval/ECE": float(ece),
            "eval/Brier": float(bri),
            "eval/OOD_AUROC": float(auroc),
            "eval/accuracy_running_mean": float(np.mean(per_ep_acc)),
            "eval/ECE_running_mean": float(np.mean(per_ep_ece)),
            "eval/AUROC_running_mean": float(np.mean(per_ep_auroc)),
        })

    pooled_probs = torch.cat(pooled_probs, dim=0)
    pooled_targets = torch.cat(pooled_targets, dim=0)
    pooled_ece = expected_calibration_error(
        pooled_probs, pooled_targets, num_bins=int(cfg.eval.ece_bins),
    )

    summary = {
        "accuracy_mean": float(np.mean(per_ep_acc)),
        "accuracy_std":  float(np.std(per_ep_acc)),
        "accuracy_ci95": float(1.96 * np.std(per_ep_acc) / np.sqrt(n_eval)),
        "adapter_type": cfg.adapter.type,
        "brier_mean":          float(np.mean(per_ep_brier)),
        "brier_std":           float(np.std(per_ep_brier)),
        "config_path": str(Path(args.config).resolve()),
        "ece_per_episode_mean": float(np.mean(per_ep_ece)),
        "ece_per_episode_std":  float(np.std(per_ep_ece)),
        "ece_pooled":          float(pooled_ece),
        "episodes_file": str(cfg.eval.episodes_file),
        "head_type": head_type,
        "n_params": int(count_trainable_params(build_model(cfg))),
        "num_episodes": int(n_eval),
        "ood_auroc_mean":      float(np.mean(per_ep_auroc)),
        "ood_auroc_std":       float(np.std(per_ep_auroc)),
        "seed": int(cfg.seed),
        "seeds_first10": [int(s) for s in seeds[:10]],
        "seeds_last10":  [int(s) for s in seeds[-10:]],
        "trainer_type": "single_episode",
    }
    return {
        "summary": summary,
        "pooled_probs": pooled_probs,
        "pooled_targets": pooled_targets,
        "last_id_scores": last_id_scores,
        "last_ood_scores": last_ood_scores,
    }


# =====================================================================
# Step 4 / Phase 2 path (prototype head, frozen meta-trained adapter)
# =====================================================================
def _fit_val_temperature(model, cfg, device, repo_root, logger) -> float:
    """Fit one global temperature T on the FROZEN val episodes (seeds from
    configs/val_episodes.yaml), Guo et al. 2017. Softmax interpretation only.
    The val seeds ([10000..10099]) are disjoint from the 600 test seeds
    ([0..599]) -> no test leakage. T is then applied unchanged to every test
    episode to produce the ts_msp OOD score + ece_ts/brier_ts calibration."""
    with open(repo_root / "configs" / "val_episodes.yaml") as f:
        val_spec = yaml.safe_load(f)
    val_seeds = list(val_spec["seeds"])
    val_split = get_cifar_fs(
        data_root=cfg.dataset.data_root,
        image_size=int(cfg.dataset.image_size), split="val",
    )
    val_iter = EpisodicIterableDataset(
        val_split, n_way=int(cfg.dataset.n_way), k_shot=int(cfg.dataset.k_shot),
        q_query=int(cfg.dataset.q_query), num_episodes=len(val_seeds),
        seed_offset=int(val_seeds[0]),
    )
    backbone = model.backbone
    logits_all, targets_all = [], []
    model.eval()
    with torch.no_grad():
        for sx, sy, qx, qy in val_iter:
            sf = backbone(sx.to(device))
            qf = backbone(qx.to(device))
            ql = model.forward_proto_from_features(sf, sy.to(device), qf)
            logits_all.append(ql.cpu())
            targets_all.append(qy.cpu())
    T = fit_temperature(torch.cat(logits_all), torch.cat(targets_all))
    logger.info(f"fit temperature on {len(val_seeds)} val episodes: T={T:.4f}")
    return T


def _evaluate_episodic(cfg, args, logger, device, wb, seeds, repo_root) -> dict:
    """Phase 2 evaluation: load the meta-trained adapter, freeze it, run
    prototype-head over the 600 test episodes."""
    n_eval = len(seeds)
    head_type = cfg.head.type
    interp = cfg.head.get("interpretation", "evidential")
    K = int(cfg.dataset.n_way)

    # --- Test split (Bertinetto 20 test classes) ----------------------
    test_split = get_cifar_fs(
        data_root=cfg.dataset.data_root,
        image_size=int(cfg.dataset.image_size),
        split="test",
    )

    # Build EpisodicIterableDataset with seed_offset = seeds[0] so the
    # 600 episodes match configs/test_episodes.yaml's seeds exactly.
    # (test_episodes.yaml uses seeds [0..599], so seed_offset=0.)
    seed_offset = int(seeds[0])
    # Sanity check: seeds must be a contiguous range starting from
    # seed_offset, since EpisodicIterableDataset increments by 1.
    expected = list(range(seed_offset, seed_offset + n_eval))
    if seeds != expected:
        raise ValueError(
            "configs/test_episodes.yaml seeds are not a contiguous "
            "range starting from seeds[0]; episodic evaluator requires "
            "contiguous seeds. Got non-contiguous seeds."
        )

    test_iter = EpisodicIterableDataset(
        test_split, n_way=K, k_shot=int(cfg.dataset.k_shot),
        q_query=int(cfg.dataset.q_query),
        num_episodes=n_eval, seed_offset=seed_offset,
    )

    # --- Load checkpoint ----------------------------------------------
    model = build_model(cfg).to(device)
    ckpt_path = args.checkpoint
    if ckpt_path is None:
        ckpt_dir = Path(cfg.output.checkpoint_dir)
        tag = f"{cfg.adapter.type}_{_head_descriptor(cfg)}_seed{cfg.seed}"
        ckpt_path = ckpt_dir / f"model_phase2_{tag}.pt"
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["state_dict"])
    best_val_epoch = int(ckpt.get("best_val_epoch", -1))
    logger.info(
        f"loaded checkpoint: {ckpt_path}  "
        f"best_val_epoch={best_val_epoch}  "
        f"best_val_acc={ckpt.get('best_val_acc', float('nan')):.3f}"
    )

    # --- OOD pools (Step 4.5 / W3): far SVHN + near CIFAR-100-heldout
    #     + optional near TinyImageNet. Each is precomputed backbone features. -
    img_size = int(cfg.dataset.image_size)
    n_ood = int(cfg.ood.num_samples)
    ood_seed = int(cfg.ood.seed)
    pools = {}
    svhn_x = get_svhn_ood(data_root=cfg.ood.data_root, image_size=img_size,
                          num_samples=n_ood, seed=ood_seed)
    pools["svhn_far"] = _extract_features(model.backbone, svhn_x, device)
    heldout_x = get_cifar_fs_heldout_ood(
        data_root=cfg.dataset.data_root, image_size=img_size,
        num_samples=n_ood, seed=ood_seed, heldout_split="val")
    pools["cifar100_near"] = _extract_features(model.backbone, heldout_x, device)
    # TinyImageNet near-OOD: from the config OR the --use-tinyimagenet CLI flag
    # (the flag lets us add it to an ALREADY-TRAINED checkpoint, eval-only, with
    # no retrain).
    use_tin = bool(cfg.ood.get("use_tinyimagenet", False)) or bool(
        getattr(args, "use_tinyimagenet", False))
    if use_tin:
        tin_x = get_tinyimagenet_ood(data_root=cfg.dataset.data_root,
                                     image_size=img_size, num_samples=n_ood,
                                     seed=ood_seed)
        pools["tin_near"] = _extract_features(model.backbone, tin_x, device)
    logger.info("OOD pools: " + ", ".join(
        f"{k}={tuple(v.shape)}" for k, v in pools.items()))

    # Temperature scaling baseline (softmax only), fit on the val episodes.
    T = None
    if interp == "softmax":
        T = _fit_val_temperature(model, cfg, device, repo_root, logger)

    prior_pc = float(cfg.loss.get("prior_per_class", 1.0))

    # --- Run the episodic evaluator (score x OOD-pool matrix) ---------
    result = evaluate_episodic(
        model=model,
        test_iterable=test_iter,
        ood_pools=pools,
        num_classes=K,
        interpretation=interp,
        ece_bins=int(cfg.eval.ece_bins),
        temperature=T,
        prior_per_class=prior_pc,
        device=device,
        logger=logger,
        wandb_run=wb,
    )
    base_summary = result["summary"]

    # Augment with config-level metadata (so the JSON schema matches
    # Step 3's + the new Phase 2 fields).
    base_summary.update({
        "adapter_type": cfg.adapter.type,
        "config_path":  str(Path(args.config).resolve()),
        "episodes_file": str(cfg.eval.episodes_file),
        "head_type": head_type,
        "interpretation": interp,
        "n_params": int(count_trainable_params(build_model(cfg))),
        "seed": int(cfg.seed),
        "seeds_first10": [int(s) for s in seeds[:10]],
        "seeds_last10":  [int(s) for s in seeds[-10:]],
        "trainer_type": "episodic",
        "best_val_epoch": int(best_val_epoch),
        "temperature": (float(T) if T is not None else 0.0),
        "prior_per_class": prior_pc,
    })
    return {
        "summary": base_summary,
        "pooled_probs": result["pooled_probs"],
        "pooled_targets": result["pooled_targets"],
        "last_id_scores": result["last_id_scores"],
        "last_ood_scores": result["last_ood_scores"],
    }


# =====================================================================
# main
# =====================================================================
def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    parser.add_argument("--checkpoint", default=None,
                        help="If omitted, derived from config "
                             "(adapter+head+seed).")
    parser.add_argument("--num-episodes", type=int, default=None,
                        help="Truncate the canonical seed list to the "
                             "first K seeds.")
    parser.add_argument("--wandb-mode", choices=["online", "offline", "disabled"],
                        default=None, help="Override cfg.wandb.mode")
    parser.add_argument("--results-suffix", default="step3",
                        help="Prefix for output JSON / PNG files in "
                             "results_dir.")
    parser.add_argument("--use-tinyimagenet", action="store_true",
                        help="Add the TinyImageNet near-OOD pool at eval time "
                             "(no retrain needed); ORs with cfg.ood.use_tinyimagenet.")
    args = parser.parse_args()

    cfg = load_config(args.config)
    logger = get_logger("bpeft.evaluate")
    repo_root = Path(__file__).resolve().parents[1]

    seeds = _load_test_seeds(repo_root, cfg)
    if args.num_episodes is not None:
        seeds = seeds[: int(args.num_episodes)]
    n_eval = len(seeds)
    trainer_type = cfg.get("trainer", {}).get("type", "single_episode") if isinstance(cfg, dict) else "single_episode"
    logger.info(
        f"config={args.config}  num_episodes={n_eval}  "
        f"trainer.type={trainer_type}  "
        f"(seeds from {cfg.eval.episodes_file})"
    )

    set_seed(int(cfg.seed))
    device = get_device()
    head_type = cfg.head.type
    K = int(cfg.dataset.n_way)

    extra_tags = [f"trainer:{trainer_type}",
                  f"phase:{'2' if trainer_type == 'episodic' else '1'}"]
    wb = _build_wandb_run(cfg, args, head_type, extra_tags=extra_tags)
    if not wb.disabled:
        logger.info(f"wandb run: {wb.run_name}  url={wb.url}")

    if trainer_type == "single_episode":
        bundle = _evaluate_finetune(cfg, args, logger, device, wb, seeds, repo_root)
    elif trainer_type == "episodic":
        bundle = _evaluate_episodic(cfg, args, logger, device, wb, seeds, repo_root)
    else:
        raise ValueError(
            f"Unknown trainer.type: {trainer_type!r}; expected "
            f"'single_episode' or 'episodic'."
        )

    summary = bundle["summary"]
    pooled_probs   = bundle["pooled_probs"]
    pooled_targets = bundle["pooled_targets"]
    last_id_scores = bundle["last_id_scores"]
    last_ood_scores = bundle["last_ood_scores"]

    # --- Final artifacts ----------------------------------------------
    out_dir = Path(cfg.output.results_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    tag = f"{args.results_suffix}_{cfg.adapter.type}_{_head_descriptor(cfg)}"
    metrics_path = out_dir / f"{tag}_metrics.json"
    rel_path     = out_dir / f"{tag}_reliability.png"
    hist_path    = out_dir / f"{tag}_ood_histogram.png"
    cm_path      = out_dir / f"{tag}_confusion_matrix.png"

    reliability_diagram(
        pooled_probs, pooled_targets, rel_path,
        num_bins=int(cfg.eval.ece_bins),
        title=f"Reliability ({head_type}, {cfg.adapter.type})  "
              f"ECE={summary['ece_pooled']:.3f}",
    )
    if last_id_scores is not None and last_ood_scores is not None:
        ood_histogram(
            last_id_scores, last_ood_scores, hist_path,
            title=f"ID vs SVHN ({head_type})  "
                  f"AUROC={summary['ood_auroc_mean']:.3f}",
        )
    confusion_matrix(
        pooled_probs, pooled_targets, cm_path, num_classes=K,
        title=f"Confusion ({head_type})  "
              f"acc={summary['accuracy_mean']:.3f}",
    )

    # sort_keys=True so a re-run on a fixed seed yields a byte-identical
    # metrics.json (Step 3 exit criterion still applies in Phase 2).
    with open(metrics_path, "w") as f:
        json.dump(summary, f, indent=2, sort_keys=True)
    logger.info(f"saved metrics: {metrics_path}")

    wb.update_summary({
        "final/accuracy_mean":  summary["accuracy_mean"],
        "final/accuracy_ci95":  summary.get("accuracy_ci95", 0.0),
        "final/f1_macro_mean":  summary.get("f1_macro_mean", 0.0),
        "final/ece_pooled":     summary["ece_pooled"],
        "final/ece_per_episode_mean": summary["ece_per_episode_mean"],
        "final/brier_mean":     summary["brier_mean"],
        "final/ood_auroc_mean": summary["ood_auroc_mean"],
        "final/fpr_at_95_tpr_mean": summary.get("fpr_at_95_tpr_mean", 1.0),
        "final/num_episodes":   summary["num_episodes"],
    })
    wb.log_image("plots/reliability_diagram", str(rel_path),
                 caption=f"{head_type} reliability")
    wb.log_image("plots/ood_histogram", str(hist_path),
                 caption=f"{head_type} ID vs OOD scores (last episode)")
    wb.log_image("plots/confusion_matrix", str(cm_path),
                 caption=f"{head_type} confusion matrix (pooled)")
    wb.log_artifact(str(metrics_path), artifact_name=f"metrics_{tag}",
                    artifact_type="metrics")
    wb.finish()

    print(json.dumps(summary, indent=2, sort_keys=True))


if __name__ == "__main__":
    main()


Overwriting scripts/evaluate.py


In [15]:
%%writefile scripts/step45_verdict.py
"""Step 4.5 verdict: master comparison table + tiered decision rule.

Consumes the two Step 4.5 metrics JSONs (evidential + softmax) produced by
scripts/evaluate.py with the score x OOD-pool matrix, and decides which tier
of the decision rule (design spec §2) was reached:

  Tier 1 (headline stands): evidential beats BOTH softmax-MSP AND
      temperature-scaled softmax on NEAR-OOD AUROC by >= margin, and is not
      meaningfully worse-calibrated (ECE) than TS-softmax.
  Tier 2 (softened): evidential ties best baseline on near-OOD but wins ID
      calibration (evid ECE < TS-softmax ECE).
  Tier 3 (reframe): neither -> "parity at a calibration cost".

The logic is pure (dict in -> result out) so it is unit-tested locally without
any GPU run; the Colab notebook and the Step 4.5 writeup both call it.
"""
from __future__ import annotations

import json
from pathlib import Path
from typing import Tuple

NEAR_MARGIN = 0.03      # AUROC margin for a "real" near-OOD win (spec §2)
ECE_TOL = 0.02          # how much worse-calibrated evidential may be and still "not worse"


def _near_pool(metrics: dict) -> str:
    """Prefer TinyImageNet near-OOD; fall back to CIFAR-100-heldout near-OOD."""
    for pool in ("tin_near", "cifar100_near"):
        if any(k.startswith(f"ood_auroc__{pool}__") for k in metrics):
            return pool
    raise KeyError("no near-OOD pool (tin_near / cifar100_near) in metrics")


def _evid_near_auroc(evid: dict, pool: str) -> float:
    return float(evid[f"ood_auroc__{pool}__vacuity"])


def _soft_near_best(soft: dict, pool: str) -> Tuple[float, str]:
    """Best near-OOD AUROC among the softmax-side scores (msp/ts_msp/energy)."""
    candidates = {}
    for score in ("msp", "ts_msp", "energy"):
        key = f"ood_auroc__{pool}__{score}"
        if key in soft:
            candidates[score] = float(soft[key])
    best_score = max(candidates, key=candidates.get)
    return candidates[best_score], best_score


def _soft_fair_ece(soft: dict) -> float:
    """Temperature-scaled ECE if available, else raw pooled ECE."""
    return float(soft.get("ece_ts", soft.get("ece_pooled")))


def decide_tier(evid: dict, soft: dict, *, near_margin: float = NEAR_MARGIN,
                ece_tol: float = ECE_TOL) -> Tuple[int, str]:
    """Return (tier in {1,2,3}, human-readable reason)."""
    pool = _near_pool(evid)
    evid_near = _evid_near_auroc(evid, pool)
    soft_near, soft_best_score = _soft_near_best(soft, pool)
    evid_ece = float(evid.get("ece_pooled"))
    soft_ece = _soft_fair_ece(soft)

    beats_near = evid_near >= soft_near + near_margin
    not_worse_cal = evid_ece <= soft_ece + ece_tol
    ties_near = abs(evid_near - soft_near) < near_margin
    wins_cal = evid_ece < soft_ece

    base = (f"near-OOD[{pool}] evid(vacuity)={evid_near:.3f} vs "
            f"best softmax {soft_best_score}={soft_near:.3f} "
            f"(margin {near_margin}); ECE evid={evid_ece:.3f} vs "
            f"TS-softmax={soft_ece:.3f} (tol {ece_tol}).")

    if beats_near and not_worse_cal:
        return 1, "TIER 1 (headline stands): " + base
    if ties_near and wins_cal:
        return 2, "TIER 2 (competitive OOD, better calibrated): " + base
    return 3, "TIER 3 (reframe to parity-at-a-calibration-cost): " + base


def build_master_table(evid: dict, soft: dict) -> str:
    """Human-readable comparison table across the score x metric grid."""
    pool = _near_pool(evid)
    rows = []
    rows.append("Step 4.5 master comparison  (5-way 5-shot CIFAR-FS, 600 episodes)")
    rows.append("=" * 72)
    rows.append(f"{'metric':<30}{'evidential':>18}{'softmax':>18}")
    rows.append("-" * 72)

    def line(label, e, s, fmt="{:.3f}"):
        es = fmt.format(e) if e is not None else "-"
        ss = fmt.format(s) if s is not None else "-"
        rows.append(f"{label:<30}{es:>18}{ss:>18}")

    line("accuracy", evid.get("accuracy_mean"), soft.get("accuracy_mean"))
    line("macro-F1", evid.get("f1_macro_mean"), soft.get("f1_macro_mean"))
    line("ECE (pooled)", evid.get("ece_pooled"), soft.get("ece_pooled"))
    line("ECE (post-TS, softmax)", None, soft.get("ece_ts"))
    line("Brier", evid.get("brier_mean"), soft.get("brier_mean"))
    line("Brier (post-TS, softmax)", None, soft.get("brier_ts"))
    rows.append("-" * 72)
    rows.append("OOD AUROC by pool x score  (higher = better):")
    for p in ("svhn_far", "cifar100_near", "tin_near"):
        ev = evid.get(f"ood_auroc__{p}__vacuity")
        for sc in ("msp", "ts_msp", "energy"):
            sv = soft.get(f"ood_auroc__{p}__{sc}")
            if ev is not None or sv is not None:
                line(f"  {p}: evid=vacuity / soft={sc}", ev, sv)
    rows.append("-" * 72)
    tier, reason = decide_tier(evid, soft)
    rows.append(reason)
    rows.append(f"NEAR-OOD pool used for the verdict: {pool}")
    return "\n".join(rows)


def _load(path: str | Path) -> dict:
    with open(path) as f:
        return json.load(f)


def main() -> None:
    import argparse
    ap = argparse.ArgumentParser()
    ap.add_argument("--evidential", required=True, help="evidential metrics JSON")
    ap.add_argument("--softmax", required=True, help="softmax metrics JSON")
    args = ap.parse_args()
    evid, soft = _load(args.evidential), _load(args.softmax)
    print(build_master_table(evid, soft))


if __name__ == "__main__":
    main()


Overwriting scripts/step45_verdict.py


In [16]:
%%writefile scripts/step45_val_sweep.py
"""Step 4.5 / W2 — VAL-only R-EDL hyperparameter sweep (in-process).

Trains a SHORT evidential model per (kl_weight_max, use_variance, prior_per_class)
combo, evaluates on the FROZEN VAL episodes (val split + configs/val_episodes.yaml
seeds), and ranks by VAL calibration (pooled ECE). Writes the winning combo into
a swept config so the subsequent FULL train+test run uses a VAL-selected operating
point instead of hand-picked defaults.

Selection is on VAL only — the 600 test seeds (configs/test_episodes.yaml) are
never touched here, so this cannot leak into the reported result.

Runs everything IN-PROCESS (no subprocess) so Colab cannot swallow the per-combo
output the way the old inline sweep cell did.

Reasoning (thesis instructions): closes the Step 4.5 §5 caveat that the retuned
R-EDL hyperparameters were reasoned defaults, not an empirically-swept choice.
The short-epoch ranking is a proxy for the full run; the winner is then trained
to convergence separately (standard sweep-then-refit protocol) — a documented
approximation, not a claim that 8-epoch ranking == 30-epoch ranking.
"""
from __future__ import annotations
import argparse
import itertools
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

import torch
import yaml

from src.utils import set_seed, get_device, load_config, get_logger
from src.datasets import get_cifar_fs, get_svhn_ood, EpisodicIterableDataset
from src.models import build_model
from src.trainers import EpisodicTrainer
from src.evaluators import evaluate_episodic


def _extract_features(backbone, x, device, batch_size=64):
    backbone.eval()
    chunks = []
    with torch.no_grad():
        for i in range(0, len(x), batch_size):
            chunks.append(backbone(x[i:i + batch_size].to(device)).cpu())
    return torch.cat(chunks, dim=0)


def _train_and_val(cfg, epochs, device, svhn_feats, logger=None):
    """Train `epochs` epochs, then evaluate on the VAL episodes. Returns a
    dict of VAL metrics (ECE / acc / far-OOD AUROC). No test data touched.

    `logger` (if given) is passed to the trainer so each combo prints per-epoch
    progress — otherwise a long combo looks like a hang on Colab."""
    interp = cfg.head.get("interpretation", "evidential")
    n_way = int(cfg.dataset.n_way)
    k_shot = int(cfg.dataset.k_shot)
    q_query = int(cfg.dataset.q_query)
    eps = int(cfg.trainer.episodes_per_epoch)
    veps = int(cfg.trainer.val_episodes_per_epoch)
    tso = int(cfg.trainer.train_seed_offset)

    train_split = get_cifar_fs(data_root=cfg.dataset.data_root,
                               image_size=int(cfg.dataset.image_size), split="train")
    val_split = get_cifar_fs(data_root=cfg.dataset.data_root,
                             image_size=int(cfg.dataset.image_size), split="val")
    repo = Path(__file__).resolve().parents[1]
    val_seeds = yaml.safe_load(open(repo / cfg.eval.val_episodes_file))["seeds"]
    vso = int(val_seeds[0])

    def train_factory(epoch):
        return EpisodicIterableDataset(
            train_split, n_way=n_way, k_shot=k_shot, q_query=q_query,
            num_episodes=eps, seed_offset=tso + (epoch - 1) * eps)

    def val_factory(epoch):
        return EpisodicIterableDataset(
            val_split, n_way=n_way, k_shot=k_shot, q_query=q_query,
            num_episodes=veps, seed_offset=vso)

    model = build_model(cfg).to(device)
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad],
                           lr=float(cfg.train.lr),
                           weight_decay=float(cfg.train.weight_decay))
    trainer = EpisodicTrainer(
        model=model, optimizer=opt, num_classes=n_way, num_epochs=epochs,
        episodes_per_epoch=eps, val_episodes_per_epoch=veps,
        early_stop_patience=int(cfg.trainer.early_stop_patience),
        interpretation=interp,
        kl_weight_max=float(cfg.loss.kl_weight_max),
        kl_anneal_steps=int(cfg.loss.kl_anneal_steps),
        ece_bins=int(cfg.eval.ece_bins), logger=logger, wandb_run=None,
        device=device, collapse_threshold=float(cfg.trainer.collapse_threshold),
        evid_prior_per_class=float(cfg.loss.get("prior_per_class", 1.0)),
        evid_use_variance=bool(cfg.loss.get("use_variance", True)))
    trainer.fit(train_factory, val_factory)

    val_iter = EpisodicIterableDataset(
        val_split, n_way=n_way, k_shot=k_shot, q_query=q_query,
        num_episodes=veps, seed_offset=vso)
    res = evaluate_episodic(
        model, val_iter, {"svhn_far": svhn_feats}, num_classes=n_way,
        interpretation=interp, ece_bins=int(cfg.eval.ece_bins),
        temperature=None, prior_per_class=float(cfg.loss.get("prior_per_class", 1.0)),
        device=device)
    s = res["summary"]
    return {"val_ece": float(s["ece_pooled"]),
            "val_acc": float(s["accuracy_mean"]),
            "val_svhn_auroc": float(s.get("ood_auroc__svhn_far__vacuity", float("nan")))}


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--base-config", default="configs/exp_phase2_evidential_retuned.yaml")
    ap.add_argument("--out-config", default="configs/exp_phase2_evidential_swept.yaml")
    ap.add_argument("--epochs", type=int, default=8,
                    help="Short training budget per combo (proxy for the full run).")
    ap.add_argument("--kl", type=float, nargs="+", default=[0.05, 0.1, 0.25])
    ap.add_argument("--use-variance", type=int, nargs="+", default=[0, 1],
                    help="0=drop variance term (R-EDL), 1=keep (Sensoy).")
    ap.add_argument("--prior", type=float, nargs="+", default=[1.0])
    ap.add_argument("--acc-tol", type=float, default=0.03,
                    help="Winner must be within this val-acc of the best combo.")
    args = ap.parse_args()

    logger = get_logger("bpeft.sweep")
    device = get_device()
    base = load_config(args.base_config)

    # Far-OOD (SVHN) backbone features: config-independent (frozen backbone),
    # so extract ONCE and reuse for every combo's val eval.
    set_seed(int(base.seed))
    m0 = build_model(base).to(device)
    svhn_x = get_svhn_ood(data_root=base.ood.data_root,
                          image_size=int(base.dataset.image_size),
                          num_samples=int(base.ood.num_samples), seed=int(base.ood.seed))
    svhn_feats = _extract_features(m0.backbone, svhn_x, device)
    del m0

    results = []
    for kl, uv, pr in itertools.product(args.kl, args.use_variance, args.prior):
        set_seed(int(base.seed))  # same init per combo -> fair comparison
        cfg = load_config(args.base_config)
        cfg.loss.kl_weight_max = float(kl)
        cfg.loss.use_variance = bool(uv)
        cfg.loss.prior_per_class = float(pr)
        logger.info(f"=== combo: kl={kl} use_variance={bool(uv)} prior={pr} "
                    f"({args.epochs} epochs) ===")
        m = _train_and_val(cfg, args.epochs, device, svhn_feats, logger=logger)
        m.update({"kl": float(kl), "use_variance": bool(uv), "prior": float(pr)})
        results.append(m)
        logger.info(f"kl={kl} var={bool(uv)} prior={pr} -> "
                    f"val_ECE={m['val_ece']:.4f} val_acc={m['val_acc']:.4f} "
                    f"val_svhn_AUROC={m['val_svhn_auroc']:.4f}")

    best_acc = max(r["val_acc"] for r in results)
    eligible = [r for r in results if r["val_acc"] >= best_acc - args.acc_tol]
    winner = min(eligible, key=lambda r: r["val_ece"])

    print("\n=== VAL sweep (selection on VAL only; test never touched) ===")
    print(f"{'kl':>6}{'var':>7}{'prior':>7}{'val_ECE':>10}{'val_acc':>10}{'val_AUROC':>11}")
    for r in sorted(results, key=lambda r: r["val_ece"]):
        mark = "  <== winner (lowest val ECE within acc tol)" if r is winner else ""
        print(f"{r['kl']:>6}{str(r['use_variance']):>7}{r['prior']:>7}"
              f"{r['val_ece']:>10.4f}{r['val_acc']:>10.4f}"
              f"{r['val_svhn_auroc']:>11.4f}{mark}")

    out = {
        "extends": Path(args.base_config).name,
        "_note": "Step 4.5 W2 — VAL-selected R-EDL config (scripts/step45_val_sweep.py). "
                 "Selected by lowest VAL pooled-ECE within acc tolerance; test untouched.",
        "loss": {"kl_weight_max": winner["kl"],
                 "use_variance": winner["use_variance"],
                 "prior_per_class": winner["prior"]},
    }
    with open(args.out_config, "w") as f:
        yaml.safe_dump(out, f, sort_keys=False)
    print(f"\nwrote {args.out_config}: kl_weight_max={winner['kl']} "
          f"use_variance={winner['use_variance']} prior_per_class={winner['prior']} "
          f"(val_ECE={winner['val_ece']:.4f}, val_acc={winner['val_acc']:.4f})")


if __name__ == "__main__":
    main()


Overwriting scripts/step45_val_sweep.py


## 2. Canonical Bertinetto split (must NOT be the synthetic fallback)

In [17]:
import os

script = "scripts/build_cifar_fs_split.py"
if os.path.exists(script):
    get_ipython().system(f"python {script}")
else:
    print(f"WARNING: {script} not found in this Drive copy — run the Materialize "
          f"section (1b) above first. Continuing with the existing split file.")

import json
split_path = "data/cifar_fs_split.json"
assert os.path.exists(split_path), (
    f"{split_path} does not exist and {script} could not run — run section 1b, "
    f"or run notebooks/step4_episodic.ipynb's fetch cell first."
)
st = json.load(open(split_path))["_status"]
# Semantic check (mirrors src/datasets/cifar_fs.py): any non-fallback status is OK.
assert st != "synthetic_fallback", f"split is {st!r} (synthetic fallback) — aborting"
print("split OK:", st)

wrote /content/drive/MyDrive/bpeft_step4/thesis/data/cifar_fs_split.json  (64/16/20, disjoint, union=100, status=canonical_bertinetto_via_torchmeta)
split OK: canonical_bertinetto_via_torchmeta


## 3. Pre-flight — confirm materialized code imports

In [18]:
# --- Pre-flight: confirm the Step 4.5 code materialized correctly ---------
import importlib, inspect, torch
for m in ["src.evaluators", "src.losses", "src.datasets", "src.trainers",
          "src.evaluators.episodic"]:
    importlib.reload(importlib.import_module(m))
from src.evaluators import (fit_temperature, apply_temperature, energy_score,
                            evaluate_episodic, evidence_to_probs_and_vacuity)
from src.losses import evidential_mse_loss
from src.datasets import get_cifar_fs_heldout_ood, get_tinyimagenet_ood

T = fit_temperature(torch.randn(200, 5) * 4.0, torch.randint(0, 5, (200,)))
assert T > 0
assert energy_score(torch.randn(8, 5)).shape == (8,)
assert "prior_per_class" in inspect.signature(evidential_mse_loss).parameters
assert "ood_pools" in inspect.signature(evaluate_episodic).parameters
print("Step 4.5 code OK: temperature + energy + R-EDL + score x OOD matrix all "
      "present.  T_smoke=%.3f" % T)

Step 4.5 code OK: temperature + energy + R-EDL + score x OOD matrix all present.  T_smoke=3.663


## 4. Softmax baseline — train + eval (TS + energy + far/near-OOD)

In [19]:
# Softmax baseline: train + eval. --use-tinyimagenet adds the TinyImageNet
# near-OOD pool (heavy download ~240MB the FIRST time, then cached on Drive).
!python scripts/train.py    --config configs/exp_phase2_softmax.yaml
!python scripts/evaluate.py --config configs/exp_phase2_softmax.yaml \
    --num-episodes 600 --wandb-mode disabled --results-suffix step45 --use-tinyimagenet

[08:07:24] INFO bpeft.train: config: configs/exp_phase2_softmax.yaml  seed: 42  trainer.type: episodic
[08:07:24] INFO bpeft.train: wandb: disabled (no-op logger)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 186MB/s]
[08:07:41] INFO bpeft.train: trainable params: 16,912
[08:08:40] INFO bpeft.train: epoch   1/30  train_loss=0.4700  train_acc=0.855  val_loss=0.6230  val_acc=0.780  kl_w=0.000  mean_ev=0.0000  grad_norm=0.5997  global_step=100
[08:09:36] INFO bpeft.train: epoch   2/30  train_loss=0.4341  train_acc=0.846  val_loss=0.5852  val_acc=0.795  kl_w=0.000  mean_ev=0.0000  grad_norm=0.5992  global_step=200
[08:10:33] INFO bpeft.train: epoch   3/30  train_loss=0.3905  train_acc=0.865  val_loss=0.5856  val_acc=0.786  kl_w=0.000  mean_ev=0.0000  grad_norm=0.6480  global_step=300
[08:11:29] INFO bpeft.train: epoch   4/30  train_loss=0.3989  train_acc=0.862  val_loss=0.5

## 5. VAL-only R-EDL sweep → writes the swept evidential config

In [20]:
# VAL-only R-EDL sweep: selection is on the frozen VAL episodes; the 600 TEST
# seeds are never touched. Writes configs/exp_phase2_evidential_swept.yaml with
# the lowest-VAL-ECE hyperparameters. Default = 3 combos x 8 epochs (~25 min on
# a T4). Add `1` to --use-variance for the full 6-combo grid; lower --epochs to
# go faster (ranking is a proxy for the full run anyway).
!python scripts/step45_val_sweep.py \
    --base-config configs/exp_phase2_evidential_retuned.yaml \
    --out-config  configs/exp_phase2_evidential_swept.yaml \
    --epochs 8 --kl 0.05 0.1 0.25 --use-variance 0

[08:19:09] INFO bpeft.sweep: === combo: kl=0.05 use_variance=False prior=1.0 (8 epochs) ===
[08:20:14] INFO bpeft.sweep: epoch   1/8  train_loss=0.4329  train_acc=0.849  val_loss=0.5131  val_acc=0.787  kl_w=0.005  mean_ev=2.0980  grad_norm=0.4158  global_step=100
[08:21:12] INFO bpeft.sweep: epoch   2/8  train_loss=0.3889  train_acc=0.836  val_loss=0.5090  val_acc=0.794  kl_w=0.010  mean_ev=2.1836  grad_norm=0.3953  global_step=200
[08:22:10] INFO bpeft.sweep: epoch   3/8  train_loss=0.3675  train_acc=0.848  val_loss=0.5017  val_acc=0.798  kl_w=0.015  mean_ev=2.4669  grad_norm=0.4819  global_step=300
[08:23:08] INFO bpeft.sweep: epoch   4/8  train_loss=0.3625  train_acc=0.846  val_loss=0.5131  val_acc=0.777  kl_w=0.020  mean_ev=2.4818  grad_norm=0.5075  global_step=400
[08:24:06] INFO bpeft.sweep: epoch   5/8  train_loss=0.3506  train_acc=0.858  val_loss=0.4662  val_acc=0.794  kl_w=0.025  mean_ev=2.6154  grad_norm=0.5517  global_step=500
[08:25:05] INFO bpeft.sweep: epoch   6/8  train_

## 6. Evidential (VAL-selected) — train + eval

In [21]:
# Full train+eval of the VAL-selected evidential config. Falls back to the
# reasoned-default retuned config if the sweep cell was skipped.
import os
cfg = ('configs/exp_phase2_evidential_swept.yaml'
       if os.path.exists('configs/exp_phase2_evidential_swept.yaml')
       else 'configs/exp_phase2_evidential_retuned.yaml')
print('using config:', cfg)
get_ipython().system(f'python scripts/train.py    --config {cfg}')
get_ipython().system(f'python scripts/evaluate.py --config {cfg} '
                     f'--num-episodes 600 --wandb-mode disabled '
                     f'--results-suffix step45 --use-tinyimagenet')

using config: configs/exp_phase2_evidential_swept.yaml
[08:44:17] INFO bpeft.train: config: configs/exp_phase2_evidential_swept.yaml  seed: 42  trainer.type: episodic
[08:44:17] INFO bpeft.train: wandb: disabled (no-op logger)
[08:44:21] INFO bpeft.train: trainable params: 16,914
[08:45:19] INFO bpeft.train: epoch   1/30  train_loss=0.4329  train_acc=0.849  val_loss=0.5131  val_acc=0.787  kl_w=0.005  mean_ev=2.0980  grad_norm=0.4158  global_step=100
[08:46:17] INFO bpeft.train: epoch   2/30  train_loss=0.3889  train_acc=0.836  val_loss=0.5090  val_acc=0.794  kl_w=0.010  mean_ev=2.1836  grad_norm=0.3953  global_step=200
[08:47:14] INFO bpeft.train: epoch   3/30  train_loss=0.3675  train_acc=0.848  val_loss=0.5017  val_acc=0.798  kl_w=0.015  mean_ev=2.4669  grad_norm=0.4819  global_step=300
[08:48:11] INFO bpeft.train: epoch   4/30  train_loss=0.3625  train_acc=0.846  val_loss=0.5131  val_acc=0.777  kl_w=0.020  mean_ev=2.4818  grad_norm=0.5075  global_step=400
[08:49:09] INFO bpeft.train

## 7. Verdict — master table + tier

In [22]:
ev = 'results/step45_bottleneck_prototype-evidential_metrics.json'
so = 'results/step45_bottleneck_prototype-softmax_metrics.json'
!python scripts/step45_verdict.py --evidential {ev} --softmax {so}

Step 4.5 master comparison  (5-way 5-shot CIFAR-FS, 600 episodes)
metric                                evidential           softmax
------------------------------------------------------------------------
accuracy                                   0.884             0.875
macro-F1                                   0.882             0.873
ECE (pooled)                               0.285             0.082
ECE (post-TS, softmax)                         -             0.041
Brier                                      0.285             0.190
Brier (post-TS, softmax)                       -             0.181
------------------------------------------------------------------------
OOD AUROC by pool x score  (higher = better):
  svhn_far: evid=vacuity / soft=msp             0.914             0.838
  svhn_far: evid=vacuity / soft=ts_msp             0.914             0.828
  svhn_far: evid=vacuity / soft=energy             0.914             0.905
  cifar100_near: evid=vacuity / soft=msp           

## 8. Persist results + swept config back to Drive

In [23]:
!cp results/step45_*_metrics.json /content/drive/MyDrive/bpeft_step4/thesis/results/ 2>/dev/null
!cp configs/exp_phase2_evidential_swept.yaml /content/drive/MyDrive/bpeft_step4/thesis/configs/ 2>/dev/null
print('done — paste the verdict table above into step_writeups/step4_5.txt §4.')

done — paste the verdict table above into step_writeups/step4_5.txt §4.
